# Reproducing Meta-RL Learning Curves on HalfCheetahVel

## VariBAD, MAML, and RL2

> **You may change, replace, split, merge, reorder, or delete any cell in this notebook.**
>
> The provided structure is only a starting point. Your implementation may use different classes,
> helper functions, optimizers, data structures, or training loops, provided that you satisfy the
> experimental protocol and clearly document every departure from the papers.

Your goal is to implement three meta-reinforcement-learning algorithms in **PyTorch**, using the
papers as the primary specifications:

1. **VariBAD**
2. **MAML for reinforcement learning**
3. **RL2**

You will then reproduce the result in **Figures 4 and 6** of the VariBAD paper for the **HalfCheetahVel** task.

## Required papers

Read the papers before implementing the methods.

### VariBAD

**Zintgraf et al., "VariBAD: A Very Good Method for Bayes-Adaptive Deep RL via Meta-Learning," ICLR 2020**

- Paper: https://arxiv.org/abs/1910.08348

### RL2

**Duan et al., "RL2: Fast Reinforcement Learning via Slow Reinforcement Learning," 2016**

- Paper: https://arxiv.org/abs/1611.02779

### MAML

**Finn, Abbeel, and Levine, "Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks," ICML 2017**

- Paper: https://arxiv.org/abs/1703.03400

### Important difference

The published VariBAD Figure 6 compares against **E-MAML**, not plain MAML. This assignment asks
you to implement **vanilla MAML** from the MAML paper.

### Figure 4 reproduction note

The published Figure 4 reports **average meta-test return over the first five rollouts** for each
MuJoCo environment. In this notebook, reproduce only the **HalfCheetahVel** panel and compare the
three required implementations: VariBAD, RL2, and vanilla MAML.

As with Figure 6, this is not an exact method-for-method reproduction of the published panel because
the paper reports **E-MAML**, while this assignment requires **vanilla MAML**. Clearly label the
result as a Figure 4-style HalfCheetahVel reproduction.


## Learning objectives

By completing this assignment, you should be able to:

- distinguish optimization-based, recurrent, and inference-based meta-RL;
- implement a hidden-task continuous-control benchmark;
- separate meta-training tasks from held-out meta-test tasks;
- implement within-task adaptation without leaking the task identity.

## HalfCheetahVel protocol

Each task is defined by a hidden target velocity $v^\star$. The agent observes the standard
HalfCheetah state but **does not observe $v^\star$**.

At each time step, use

$$
r_t = -\left|v_t-v^\star\right| - 0.05\lVert a_t\rVert_2^2,
$$

where $v_t$ is the forward velocity and $a_t$ is the action.

### Required benchmark settings

- Target velocities are sampled from $[0,3]$.
- One MDP episode has horizon $H=200$.
- VariBAD and RL2 are trained on two consecutive episodes of the same task:
  $$
  H^+ = 2H = 400.
  $$
- The physical environment resets after the first episode, but:
  - RL2 must preserve its recurrent hidden state;
  - VariBAD must preserve its inferred task posterior;
  - a done/reset indicator must be made available to recurrent inference.
- MAML adapts through gradient updates computed from support trajectories and is evaluated on a
  fresh query trajectory.
- Meta-test tasks must not be used for gradient updates during meta-training.
- The target velocity or task ID must never be passed directly to the policy, encoder, critic, or
  value function.

## What must be submitted

1. This completed notebook.
2. Implementations of VariBAD, MAML, and RL2.
3. Training curves.

You may checkpoint models during training, but the final plots must be generated by code in this
notebook.

### Required figure outputs

The completed notebook must generate both of the following for **HalfCheetahVel only**:

1. A **Figure 6-style** two-panel learning-curve plot versus meta-training environment frames,
   showing first-rollout and adapted/method-specific final-rollout return.
2. A **Figure 4-style** plot showing average test return for rollouts 1 through 5 at the selected
   final checkpoint.

Both figures must be generated from the actual trained models, not from manually entered values.


In [ ]:
# Colab setup.
# Run this cell once in a fresh runtime, then restart the runtime only if Colab requests it.

%pip install -q "gymnasium[mujoco]>=1.0,<2" "mujoco>=3.2,<4" \
    "numpy>=1.24" "pandas>=2.0" "matplotlib>=3.7" "tqdm>=4.66"

In [ ]:
# Imports used by the three implementations and the provided plotting utilities.

import copy
import math
import os
import random
import time
from collections import OrderedDict, deque
from dataclasses import asdict, dataclass, fields
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal, kl_divergence
from tqdm.auto import tqdm

try:
    # PyTorch 2.x
    from torch.func import functional_call
except ImportError:  # pragma: no cover - compatibility with older Colab images
    from torch.nn.utils.stateless import functional_call


In [ ]:
@dataclass
class ExperimentConfig:
    # Environment and task distribution
    env_id: str = "HalfCheetah-v5"
    velocity_low: float = 0.0
    velocity_high: float = 3.0
    episode_horizon: int = 200
    recurrent_rollouts_per_task: int = 2

    # Task splits
    num_train_tasks: int = 100
    num_validation_tasks: int = 20
    num_test_tasks: int = 30
    task_split_seed: int = 2026

    # Full reproduction target
    total_frames: int = 100_000_000
    evaluation_interval_frames: int = 1_000_000
    evaluation_tasks: int = 30
    evaluation_episodes_per_task: int = 1
    seeds: Tuple[int, ...] = (11, 22, 33, 44, 55)

    # Shared RL defaults. Each method may override these.
    gamma: float = 0.97
    gae_lambda: float = 0.90
    max_grad_norm: float = 0.5

    # Logging
    verbose_interval_updates: int = 10


FULL_CONFIG = ExperimentConfig()

# A small configuration for debugging interfaces only.
DEBUG_CONFIG = ExperimentConfig(
    num_train_tasks=8,
    num_validation_tasks=4,
    num_test_tasks=4,
    total_frames=20_000,
    evaluation_interval_frames=5_000,
    evaluation_tasks=4,
    seeds=(0,),
)

# Change this during development.
CONFIG = DEBUG_CONFIG

print(asdict(CONFIG))

In [ ]:
# Figure 4 evaluates five consecutive meta-test rollouts at the selected checkpoint.
# This is kept separate from the existing configuration fields so the original
# experiment configuration remains unchanged.
FIGURE4_EVALUATION_ROLLOUTS: int = 5


In [ ]:
def seed_everything(seed: int) -> None:
    """Seed Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Provided environment infrastructure

The following wrapper defines the common benchmark. You may modify its implementation, but all
methods must ultimately be evaluated with the same reward, task split, horizon, and observation
semantics.

The standard Gymnasium HalfCheetah observation does not expose the global torso $x$-position.
The wrapper recomputes the reward using the hidden target velocity.

In [ ]:
class HalfCheetahVel(gym.Wrapper):
    """HalfCheetah with a hidden target-velocity task."""

    def __init__(
        self,
        target_velocity: float,
        env_id: str = "HalfCheetah-v5",
        horizon: int = 200,
        render_mode: Optional[str] = None,
    ):
        base_env = gym.make(
            env_id,
            render_mode=render_mode,
            max_episode_steps=horizon,
        )
        super().__init__(base_env)
        self.target_velocity = float(target_velocity)
        self.horizon = int(horizon)
        self._elapsed_steps_local = 0

    @property
    def task(self) -> Dict[str, float]:
        # This property is for experiment control and logging only.
        # Do not feed it to an agent.
        return {"velocity": self.target_velocity}

    def reset(
        self,
        *,
        seed: Optional[int] = None,
        options: Optional[dict] = None,
    ):
        self._elapsed_steps_local = 0
        obs, info = self.env.reset(seed=seed, options=options)
        info = dict(info)
        info["task_velocity_for_logging_only"] = self.target_velocity
        return obs.astype(np.float32), info

    def step(self, action):
        x_before = float(self.unwrapped.data.qpos[0])
        _, _, terminated, truncated, info = self.env.step(action)
        x_after = float(self.unwrapped.data.qpos[0])

        forward_velocity = (x_after - x_before) / float(self.unwrapped.dt)
        velocity_reward = -abs(forward_velocity - self.target_velocity)
        control_cost = 0.05 * float(np.square(action).sum())
        reward = velocity_reward - control_cost

        self._elapsed_steps_local += 1
        obs = self.unwrapped._get_obs().astype(np.float32)

        info = dict(info)
        info.update(
            {
                "forward_velocity": forward_velocity,
                "velocity_error": forward_velocity - self.target_velocity,
                "reward_velocity": velocity_reward,
                "reward_control": -control_cost,
                "task_velocity_for_logging_only": self.target_velocity,
            }
        )
        return obs, float(reward), bool(terminated), bool(truncated), info

In [ ]:
@dataclass(frozen=True)
class TaskSplits:
    train: np.ndarray
    validation: np.ndarray
    test: np.ndarray


def make_task_splits(config: ExperimentConfig) -> TaskSplits:
    """Create deterministic, disjoint finite task sets."""
    rng = np.random.default_rng(config.task_split_seed)
    total = config.num_train_tasks + config.num_validation_tasks + config.num_test_tasks
    velocities = rng.uniform(
        config.velocity_low,
        config.velocity_high,
        size=total,
    ).astype(np.float32)

    n_train = config.num_train_tasks
    n_val = config.num_validation_tasks

    return TaskSplits(
        train=velocities[:n_train],
        validation=velocities[n_train:n_train + n_val],
        test=velocities[n_train + n_val:],
    )


TASKS = make_task_splits(CONFIG)
print("Training tasks:", TASKS.train)
print("Validation tasks:", TASKS.validation)
print("Test tasks:", TASKS.test)

In [ ]:
# Environment smoke test

seed_everything(0)
test_env = HalfCheetahVel(
    target_velocity=float(TASKS.train[0]),
    env_id=CONFIG.env_id,
    horizon=CONFIG.episode_horizon,
)
obs, info = test_env.reset(seed=0)
action = test_env.action_space.sample()
next_obs, reward, terminated, truncated, info = test_env.step(action)

print("Observation shape:", obs.shape)
print("Action shape:", action.shape)
print("One-step reward:", reward)
print("Forward velocity:", info["forward_velocity"])
print("Target velocity:", info["task_velocity_for_logging_only"])

assert obs.ndim == 1
assert next_obs.shape == obs.shape
assert np.isfinite(reward)
test_env.close()

## Common result contract

You may organize training however you want, but each completed run must be converted to the
following in-memory format so that the provided plotting tools can compare methods.

```python
history = {
    "method": "VariBAD",               # or "RL2", "MAML"
    "seed": 11,
    "frames": np.ndarray,              # shape [K], monotonically increasing
    "first_rollout_return": np.ndarray,# shape [K]
    "adapted_return": np.ndarray,      # shape [K]
    "metadata": {...},                 # hyperparameters and protocol
}
```

Interpretation:

- `first_rollout_return`: mean return on the first episode of held-out tasks, before useful
  task-specific information is available.
- `adapted_return`:
  - VariBAD: second rollout while retaining the posterior;
  - RL2: second rollout while retaining the recurrent state;
  - MAML: query rollout after the declared support-set gradient adaptation.

All evaluation must use the same fixed held-out task set for a given seed and checkpoint.

### Figure 4 result contract

The Figure 4-style evaluation requires per-seed, per-task, per-rollout returns at the selected final
checkpoint. Store them as:

```python
figure4_rollout_results = {
    "VariBAD": np.ndarray,  # [num_seeds, num_test_tasks, 5]
    "RL2": np.ndarray,      # [num_seeds, num_test_tasks, 5]
    "MAML": np.ndarray,     # [num_seeds, num_test_tasks, 5]
}
```

For VariBAD and RL2, do not reset adaptation state between the five rollouts of a task. For MAML,
retain only the task-specific fast parameters between rollouts and reset them to the learned
meta-initialization when moving to a new task. Preserve the individual task dimension until after all
evaluation data have been collected.


In [ ]:
def validate_history(history: Mapping[str, Any]) -> None:
    """Validate the output format expected by the plotting utilities."""
    required = {
        "method",
        "seed",
        "frames",
        "first_rollout_return",
        "adapted_return",
        "metadata",
    }
    missing = required.difference(history.keys())
    if missing:
        raise KeyError(f"History is missing keys: {sorted(missing)}")

    frames = np.asarray(history["frames"], dtype=np.float64)
    first = np.asarray(history["first_rollout_return"], dtype=np.float64)
    adapted = np.asarray(history["adapted_return"], dtype=np.float64)

    if not (frames.ndim == first.ndim == adapted.ndim == 1):
        raise ValueError("frames and return arrays must all be one-dimensional.")
    if not (len(frames) == len(first) == len(adapted)):
        raise ValueError("frames and return arrays must have equal lengths.")
    if len(frames) == 0:
        raise ValueError("history may not be empty.")
    if np.any(np.diff(frames) <= 0):
        raise ValueError("frames must be strictly increasing.")
    if not np.all(np.isfinite(first)) or not np.all(np.isfinite(adapted)):
        raise ValueError("Return arrays contain non-finite values.")

# Part I - Shared reinforcement-learning infrastructure

You will probably need:

- a diagonal Gaussian policy for continuous actions;
- correct action scaling or clipping;
- log-probabilities under the behavior policy;
- a value function;
- trajectory storage;
- return and advantage computation;
- generalized advantage estimation;
- an on-policy optimizer such as PPO or TRPO;
- checkpoint and evaluation utilities;
- careful treatment of time-limit truncation.

These helpers may be shared across methods. Do not accidentally detach tensors needed by MAML's
meta-gradient, and do not shuffle recurrent sequences in a way that destroys temporal order.

In [ ]:
@dataclass
class TrajectoryBatch:
    """
    One ordered feed-forward episode.

    Shapes:
        observations:      [T, observation_dim]
        actions:           [T, action_dim]
        rewards:           [T]
        next_observations: [T, observation_dim]
        terminated:        [T] bool -- true MDP termination
        truncated:         [T] bool -- time-limit truncation
        dones_for_agent:   [T] bool -- physical episode boundary
        log_probs:         [T] or None
        values:            [T] or None
        returns:           [T] or None
        advantages:        [T] or None

    T is the number of transitions in this physical episode (normally 200).
    """
    observations: torch.Tensor
    actions: torch.Tensor
    rewards: torch.Tensor
    next_observations: torch.Tensor
    terminated: torch.Tensor
    truncated: torch.Tensor
    dones_for_agent: torch.Tensor
    log_probs: Optional[torch.Tensor] = None
    values: Optional[torch.Tensor] = None
    returns: Optional[torch.Tensor] = None
    advantages: Optional[torch.Tensor] = None


class SquashedDiagGaussian:
    """
    Diagonal Gaussian followed by tanh and optional action scaling.

    HalfCheetah-v5 uses [-1, 1] actions, but keeping the scale explicit makes
    the policy safe if the action-space magnitude changes.
    """

    def __init__(
        self,
        mean: torch.Tensor,
        log_std: torch.Tensor,
        action_scale: torch.Tensor | float = 1.0,
        eps: float = 1e-6,
    ):
        self.mean = mean
        self.log_std = log_std.clamp(-5.0, 2.0)
        self.std = self.log_std.exp()
        self.base_dist = Normal(self.mean, self.std)
        self.action_scale = torch.as_tensor(
            action_scale, dtype=mean.dtype, device=mean.device
        )
        self.eps = float(eps)

    def rsample(self) -> torch.Tensor:
        pre_tanh = self.base_dist.rsample()
        return torch.tanh(pre_tanh) * self.action_scale

    def sample(self) -> torch.Tensor:
        with torch.no_grad():
            pre_tanh = self.base_dist.sample()
            return torch.tanh(pre_tanh) * self.action_scale

    @property
    def mode(self) -> torch.Tensor:
        return torch.tanh(self.mean) * self.action_scale

    def log_prob(self, action: torch.Tensor) -> torch.Tensor:
        # Invert tanh with a numerically stable atanh and include the Jacobian.
        y = (action / self.action_scale).clamp(-1.0 + self.eps, 1.0 - self.eps)
        pre_tanh = 0.5 * (torch.log1p(y) - torch.log1p(-y))
        log_base = self.base_dist.log_prob(pre_tanh)
        log_det = torch.log(
            self.action_scale.abs() * (1.0 - y.pow(2)) + self.eps
        )
        return (log_base - log_det).sum(dim=-1)

    def entropy(self) -> torch.Tensor:
        # Exact entropy after tanh has no simple closed form.  The base-Gaussian
        # entropy is a stable exploration regularizer for PPO/PG.
        return self.base_dist.entropy().sum(dim=-1)


def _cfg_get(config: Any, key: str, default: Any = None) -> Any:
    if isinstance(config, Mapping):
        return config.get(key, default)
    return getattr(config, key, default)


def _mlp(
    input_dim: int,
    hidden_sizes: Sequence[int],
    output_dim: int,
    *,
    activation: type[nn.Module] = nn.Tanh,
) -> nn.Sequential:
    layers: List[nn.Module] = []
    last = int(input_dim)
    for width in hidden_sizes:
        layers.extend([nn.Linear(last, int(width)), activation()])
        last = int(width)
    layers.append(nn.Linear(last, int(output_dim)))
    return nn.Sequential(*layers)


def _cpu_state_dict(module: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        key: value.detach().cpu().clone()
        for key, value in module.state_dict().items()
    }


def _copy_trial_to_cpu(trial: Mapping[str, Any]) -> Dict[str, Any]:
    result: Dict[str, Any] = {}
    for key, value in trial.items():
        if torch.is_tensor(value):
            result[key] = value.detach().cpu()
        elif isinstance(value, np.ndarray):
            result[key] = value.copy()
        else:
            result[key] = copy.deepcopy(value)
    return result


def _concat_trajectory_batches(
    batches: Sequence[TrajectoryBatch],
) -> TrajectoryBatch:
    if not batches:
        raise ValueError("At least one trajectory is required.")

    kwargs: Dict[str, Any] = {}
    for field in fields(TrajectoryBatch):
        values = [getattr(batch, field.name) for batch in batches]
        if all(value is None for value in values):
            kwargs[field.name] = None
        elif any(value is None for value in values):
            raise ValueError(
                f"Trajectory field {field.name!r} is missing in only some batches."
            )
        else:
            kwargs[field.name] = torch.cat(values, dim=0)
    return TrajectoryBatch(**kwargs)


def _meta_trial_returns_and_advantages(
    rewards: torch.Tensor,
    values: torch.Tensor,
    gamma: float,
    gae_lambda: float,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    GAE over one complete meta-RL trial.

    rewards, values: [T].  Internal physical episode resets do *not* terminate
    the meta-trial, so value/advantage recursion is allowed to continue across
    the first reset.  The end of the final rollout is the terminal point of the
    collected meta-trial and therefore has zero bootstrap.
    """
    if rewards.ndim != 1 or values.ndim != 1 or rewards.shape != values.shape:
        raise ValueError("rewards and values must both have shape [T].")

    advantages = torch.zeros_like(rewards)
    gae = torch.zeros((), dtype=rewards.dtype, device=rewards.device)
    T = rewards.shape[0]

    for t in reversed(range(T)):
        if t == T - 1:
            next_value = torch.zeros((), dtype=values.dtype, device=values.device)
            continuation = 0.0
        else:
            next_value = values[t + 1]
            continuation = 1.0

        delta = rewards[t] + gamma * next_value - values[t]
        gae = delta + gamma * gae_lambda * continuation * gae
        advantages[t] = gae

    returns = advantages + values
    return returns, advantages


def _environment_dimensions(
    config: ExperimentConfig,
    velocity: float,
) -> Tuple[int, int, np.ndarray]:
    env = HalfCheetahVel(
        target_velocity=float(velocity),
        env_id=config.env_id,
        horizon=config.episode_horizon,
    )
    try:
        observation_dim = int(np.prod(env.observation_space.shape))
        action_dim = int(np.prod(env.action_space.shape))
        action_scale = np.asarray(env.action_space.high, dtype=np.float32)
    finally:
        env.close()
    return observation_dim, action_dim, action_scale


def _shared_agent_config(
    experiment: ExperimentConfig,
    method_config: Mapping[str, Any],
) -> Dict[str, Any]:
    cfg = {
        "env_id": experiment.env_id,
        "episode_horizon": experiment.episode_horizon,
        "gamma": experiment.gamma,
        "gae_lambda": experiment.gae_lambda,
        "max_grad_norm": experiment.max_grad_norm,
    }
    cfg.update(dict(method_config))
    return cfg


def compute_returns_and_advantages(
    rewards: torch.Tensor,
    values: torch.Tensor,
    terminated: torch.Tensor,
    truncated: torch.Tensor,
    gamma: float,
    gae_lambda: float,
    bootstrap_value: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute GAE for one ordered physical episode.

    Tensor shapes:
        rewards, values, terminated, truncated: [T]
        bootstrap_value: scalar []
        returns, advantages: [T]

    A true MDP termination suppresses value bootstrapping.  A time-limit
    truncation is *not* a terminal state, so the last transition bootstraps
    from V(s_{T}).  GAE recursion is nevertheless stopped at either physical
    episode boundary so it cannot leak into a separately reset episode.
    """
    if rewards.ndim != 1 or values.ndim != 1:
        raise ValueError("rewards and values must have shape [T].")
    if rewards.shape != values.shape:
        raise ValueError("rewards and values must have the same shape.")
    if terminated.shape != rewards.shape or truncated.shape != rewards.shape:
        raise ValueError("terminated and truncated must have shape [T].")

    terminated = terminated.to(dtype=torch.bool)
    truncated = truncated.to(dtype=torch.bool)
    bootstrap_value = torch.as_tensor(
        bootstrap_value, dtype=values.dtype, device=values.device
    ).reshape(())

    T = rewards.shape[0]
    advantages = torch.zeros_like(rewards)
    gae = torch.zeros((), dtype=rewards.dtype, device=rewards.device)

    for t in reversed(range(T)):
        if t == T - 1:
            next_value = bootstrap_value
        else:
            next_value = values[t + 1]

        # True termination removes bootstrap.  Time-limit truncation does not.
        bootstrap_mask = (~terminated[t]).to(values.dtype)
        delta = rewards[t] + gamma * bootstrap_mask * next_value - values[t]

        # Stop recursive GAE across any physical reset.  For the final
        # time-limit step the delta already contains the correct bootstrap.
        boundary = terminated[t] | truncated[t]
        continuation_mask = (~boundary).to(values.dtype)
        gae = delta + gamma * gae_lambda * continuation_mask * gae
        advantages[t] = gae

    returns = advantages + values
    return returns, advantages


In [ ]:
# Feed-forward rollout collector used by MAML.  RL2 and VariBAD use dedicated
# trial collectors because their adaptation state must survive physical resets.

def collect_episode(
    env: gym.Env,
    agent: Any,
    *,
    deterministic: bool,
    device: torch.device,
    agent_state: Optional[Any] = None,
    reset_seed: Optional[int] = None,
) -> Tuple[TrajectoryBatch, Any, float]:
    """
    Collect one physical episode in temporal order.

    ``agent_state`` is passed through for callers that want to associate
    task-specific fast parameters with the rollout.  For MAML, it is the
    mapping of fast policy parameters.  Recurrent methods deliberately use
    their own collectors so that hidden/belief state is not accidentally reset.
    """
    observation, _ = env.reset(seed=reset_seed)

    observations: List[np.ndarray] = []
    actions: List[np.ndarray] = []
    rewards: List[float] = []
    next_observations: List[np.ndarray] = []
    terminated_flags: List[bool] = []
    truncated_flags: List[bool] = []
    done_flags: List[bool] = []
    log_probs: List[float] = []
    values: List[float] = []

    episode_return = 0.0
    state = agent_state

    max_steps = int(getattr(env, "horizon", 100_000))
    for _ in range(max_steps):
        # The MAMLAgent API returns (action, log_prob, value).
        if state is None:
            output = agent.act(
                observation,
                params=None,
                deterministic=deterministic,
            )
        else:
            output = agent.act(
                observation,
                params=state,
                deterministic=deterministic,
            )

        if not isinstance(output, tuple) or len(output) < 3:
            raise TypeError(
                "Feed-forward agent.act must return "
                "(action, log_prob, value[, new_state])."
            )

        action = np.asarray(output[0], dtype=np.float32)
        log_prob = float(output[1])
        value = float(output[2])
        if len(output) >= 4:
            state = output[3]

        next_observation, reward, terminated, truncated, _ = env.step(action)
        done = bool(terminated or truncated)

        observations.append(np.asarray(observation, dtype=np.float32))
        actions.append(action)
        rewards.append(float(reward))
        next_observations.append(np.asarray(next_observation, dtype=np.float32))
        terminated_flags.append(bool(terminated))
        truncated_flags.append(bool(truncated))
        done_flags.append(done)
        log_probs.append(log_prob)
        values.append(value)

        episode_return += float(reward)
        observation = next_observation

        if done:
            break

    batch = TrajectoryBatch(
        observations=torch.as_tensor(
            np.asarray(observations), dtype=torch.float32, device=device
        ),
        actions=torch.as_tensor(
            np.asarray(actions), dtype=torch.float32, device=device
        ),
        rewards=torch.as_tensor(rewards, dtype=torch.float32, device=device),
        next_observations=torch.as_tensor(
            np.asarray(next_observations), dtype=torch.float32, device=device
        ),
        terminated=torch.as_tensor(
            terminated_flags, dtype=torch.bool, device=device
        ),
        truncated=torch.as_tensor(
            truncated_flags, dtype=torch.bool, device=device
        ),
        dones_for_agent=torch.as_tensor(
            done_flags, dtype=torch.bool, device=device
        ),
        log_probs=torch.as_tensor(log_probs, dtype=torch.float32, device=device),
        values=torch.as_tensor(values, dtype=torch.float32, device=device),
    )
    return batch, state, float(episode_return)


# Part II - MAML

## Algorithm summary

MAML learns an initialization $\theta$ such that a small policy-gradient update produces a policy
that performs well on a new task.

For a sampled task $\mathcal T_i$:

1. Collect support trajectories using $\pi_\theta$.
2. Compute a task loss $\mathcal L_{\mathcal T_i}^{\text{support}}(\theta)$.
3. Perform one or more differentiable inner updates:
   $$
   \theta_i' = \theta - \alpha \nabla_\theta
   \mathcal L_{\mathcal T_i}^{\text{support}}(\theta).
   $$
4. Collect fresh query trajectories using $\pi_{\theta_i'}$.
5. Update the shared initialization using query losses:
   $$
   \theta \leftarrow \theta - \beta \nabla_\theta
   \sum_i \mathcal L_{\mathcal T_i}^{\text{query}}(\theta_i').
   $$

In [ ]:
# MAML uses torch.func.functional_call (imported above) so that support-set
# gradient steps produce task-specific fast parameters without overwriting the
# shared meta-initialization.


In [ ]:
class MAMLPolicy(nn.Module):
    """Tanh-squashed diagonal-Gaussian policy with a functional fast forward."""

    def __init__(self, observation_dim: int, action_dim: int, **kwargs):
        super().__init__()
        hidden_sizes = tuple(kwargs.get("hidden_sizes", (64, 64)))
        action_scale = np.asarray(
            kwargs.get("action_scale", np.ones(action_dim, dtype=np.float32)),
            dtype=np.float32,
        )
        init_log_std = float(kwargs.get("init_log_std", -0.5))

        layers: List[nn.Module] = []
        last = observation_dim
        for width in hidden_sizes:
            layers.extend([nn.Linear(last, int(width)), nn.Tanh()])
            last = int(width)
        self.backbone = nn.Sequential(*layers)
        self.mean_head = nn.Linear(last, action_dim)
        self.log_std = nn.Parameter(
            torch.full((action_dim,), init_log_std, dtype=torch.float32)
        )
        self.register_buffer(
            "action_scale",
            torch.as_tensor(action_scale, dtype=torch.float32),
        )

    def forward(
        self,
        observations: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.backbone(observations)
        mean = self.mean_head(x)
        log_std = self.log_std.clamp(-5.0, 2.0).expand_as(mean)
        return mean, log_std

    def distribution(
        self,
        observations: torch.Tensor,
        params: Optional[Mapping[str, torch.Tensor]] = None,
    ) -> SquashedDiagGaussian:
        """Return the action distribution under base or task-specific parameters."""
        if params is None:
            mean, log_std = self.forward(observations)
        else:
            mean, log_std = functional_call(
                self,
                params,
                (observations,),
                strict=False,
            )
        return SquashedDiagGaussian(mean, log_std, self.action_scale)

    def act(
        self,
        observation: np.ndarray,
        *,
        params: Optional[Mapping[str, torch.Tensor]] = None,
        deterministic: bool = False,
    ) -> Tuple[np.ndarray, float]:
        """Choose one bounded action and return its behavior log probability."""
        device = next(self.parameters()).device
        obs = torch.as_tensor(
            observation, dtype=torch.float32, device=device
        ).unsqueeze(0)

        with torch.no_grad():
            dist = self.distribution(obs, params=params)
            action = dist.mode if deterministic else dist.sample()
            log_prob = dist.log_prob(action)

        return (
            action.squeeze(0).cpu().numpy().astype(np.float32),
            float(log_prob.item()),
        )


In [ ]:
class MAMLValueFunction(nn.Module):
    """Feed-forward state-value baseline used to form GAE advantages."""

    def __init__(self, observation_dim: int, **kwargs):
        super().__init__()
        hidden_sizes = tuple(kwargs.get("hidden_sizes", (64, 64)))
        self.network = _mlp(
            observation_dim,
            hidden_sizes,
            1,
            activation=nn.Tanh,
        )

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.network(observations).squeeze(-1)


In [ ]:
class MAMLAgent:
    """
    Vanilla policy-gradient MAML.

    Task-specific information is stored only in the fast policy parameters.
    With ``first_order=False`` the support update uses create_graph=True, so
    the outer query loss differentiates through the inner gradient step.
    """

    def __init__(self, observation_dim: int, action_dim: int, config: Any):
        self.config = dict(config) if isinstance(config, Mapping) else vars(config)
        self.device = torch.device(_cfg_get(config, "device", DEVICE))

        self.inner_lr = float(_cfg_get(config, "inner_lr", 0.1))
        self.inner_steps = int(_cfg_get(config, "inner_steps", 1))
        self.first_order = bool(_cfg_get(config, "first_order", False))
        self.gamma = float(_cfg_get(config, "gamma", 0.97))
        self.gae_lambda = float(_cfg_get(config, "gae_lambda", 0.90))
        self.max_grad_norm = float(_cfg_get(config, "max_grad_norm", 0.5))
        self.entropy_coef = float(_cfg_get(config, "entropy_coef", 0.0))
        self.value_epochs = int(_cfg_get(config, "value_epochs", 5))

        policy_kwargs = {
            "hidden_sizes": _cfg_get(config, "hidden_sizes", (64, 64)),
            "action_scale": _cfg_get(
                config, "action_scale", np.ones(action_dim, dtype=np.float32)
            ),
            "init_log_std": _cfg_get(config, "init_log_std", -0.5),
        }
        self.policy = MAMLPolicy(
            observation_dim, action_dim, **policy_kwargs
        ).to(self.device)
        self.value = MAMLValueFunction(
            observation_dim,
            hidden_sizes=_cfg_get(config, "value_hidden_sizes", (64, 64)),
        ).to(self.device)

        self.outer_optimizer = optim.Adam(
            self.policy.parameters(),
            lr=float(_cfg_get(config, "outer_lr", 3e-4)),
        )
        self.value_optimizer = optim.Adam(
            self.value.parameters(),
            lr=float(_cfg_get(config, "value_lr", 1e-3)),
        )

    def act(
        self,
        observation: np.ndarray,
        *,
        params: Optional[Mapping[str, torch.Tensor]] = None,
        deterministic: bool = False,
    ) -> Tuple[np.ndarray, float, float]:
        action, log_prob = self.policy.act(
            observation, params=params, deterministic=deterministic
        )
        obs = torch.as_tensor(
            observation, dtype=torch.float32, device=self.device
        ).unsqueeze(0)
        with torch.no_grad():
            value = self.value(obs)
        return action, log_prob, float(value.item())

    def prepare_batch(self, batch: TrajectoryBatch) -> TrajectoryBatch:
        """Attach bootstrapped returns and normalized GAE advantages."""
        with torch.no_grad():
            values = self.value(batch.observations)
            if bool(batch.terminated[-1].item()):
                bootstrap_value = torch.zeros((), device=self.device)
            else:
                bootstrap_value = self.value(
                    batch.next_observations[-1].unsqueeze(0)
                ).squeeze(0)

            returns, advantages = compute_returns_and_advantages(
                batch.rewards,
                values,
                batch.terminated,
                batch.truncated,
                self.gamma,
                self.gae_lambda,
                bootstrap_value,
            )
            if advantages.numel() > 1:
                advantages = (
                    advantages - advantages.mean()
                ) / (advantages.std(unbiased=False) + 1e-8)

        batch.values = values.detach()
        batch.returns = returns.detach()
        batch.advantages = advantages.detach()
        return batch

    def _policy_loss(
        self,
        batch: TrajectoryBatch,
        params: Optional[Mapping[str, torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        if batch.advantages is None:
            self.prepare_batch(batch)
        dist = self.policy.distribution(batch.observations, params=params)
        log_prob = dist.log_prob(batch.actions)
        entropy = dist.entropy().mean()
        pg_loss = -(log_prob * batch.advantages.detach()).mean()
        loss = pg_loss - self.entropy_coef * entropy
        return loss, entropy

    def adapt(
        self,
        support_batch: TrajectoryBatch,
        initial_params: Optional[Mapping[str, torch.Tensor]] = None,
        *,
        track_meta_gradient: bool = True,
    ) -> OrderedDict[str, torch.Tensor]:
        """
        Return task-adapted parameters.

        Exact MAML: create_graph=True in each inner step.
        First-order MAML: detach only the support gradient, preserving the
        identity path from the fast parameter back to the meta parameter.
        Evaluation: detach after every update so no outer graph is retained.
        """
        if support_batch.advantages is None:
            self.prepare_batch(support_batch)

        if initial_params is None:
            params: OrderedDict[str, torch.Tensor] = OrderedDict(
                self.policy.named_parameters()
            )
        else:
            params = OrderedDict(initial_params.items())

        for _ in range(self.inner_steps):
            loss, _ = self._policy_loss(support_batch, params=params)
            create_graph = bool(track_meta_gradient and not self.first_order)
            grads = torch.autograd.grad(
                loss,
                tuple(params.values()),
                create_graph=create_graph,
                retain_graph=create_graph,
            )

            updated: OrderedDict[str, torch.Tensor] = OrderedDict()
            for (name, parameter), grad in zip(params.items(), grads):
                if track_meta_gradient:
                    if self.first_order:
                        grad = grad.detach()
                    updated[name] = parameter - self.inner_lr * grad
                else:
                    updated[name] = (
                        parameter - self.inner_lr * grad
                    ).detach().requires_grad_(True)
            params = updated

        return params

    def _fit_value_function(
        self,
        batches: Sequence[TrajectoryBatch],
    ) -> float:
        usable = [batch for batch in batches if batch.returns is not None]
        if not usable:
            return 0.0
        observations = torch.cat([batch.observations for batch in usable], dim=0)
        targets = torch.cat([batch.returns for batch in usable], dim=0).detach()

        last_loss = torch.zeros((), device=self.device)
        for _ in range(self.value_epochs):
            prediction = self.value(observations)
            last_loss = F.mse_loss(prediction, targets)
            self.value_optimizer.zero_grad(set_to_none=True)
            last_loss.backward()
            nn.utils.clip_grad_norm_(
                self.value.parameters(), self.max_grad_norm
            )
            self.value_optimizer.step()
        return float(last_loss.detach().item())

    def meta_update(self, task_data: Sequence[Any]) -> Dict[str, float]:
        """Apply one outer update from fresh query data for each sampled task."""
        if not task_data:
            raise ValueError("MAML meta_update requires at least one task.")

        query_losses: List[torch.Tensor] = []
        entropies: List[torch.Tensor] = []
        value_batches: List[TrajectoryBatch] = []

        for item in task_data:
            support = item["support"]
            query = item["query"]
            if support.advantages is None:
                self.prepare_batch(support)
            if query.advantages is None:
                self.prepare_batch(query)

            fast_params = self.adapt(
                support,
                track_meta_gradient=True,
            )
            query_loss, entropy = self._policy_loss(
                query, params=fast_params
            )
            query_losses.append(query_loss)
            entropies.append(entropy)
            value_batches.extend([support, query])

        outer_loss = torch.stack(query_losses).mean()
        self.outer_optimizer.zero_grad(set_to_none=True)
        outer_loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(
            self.policy.parameters(), self.max_grad_norm
        )
        self.outer_optimizer.step()

        value_loss = self._fit_value_function(value_batches)
        return {
            "outer_loss": float(outer_loss.detach().item()),
            "value_loss": value_loss,
            "entropy": float(torch.stack(entropies).mean().detach().item()),
            "grad_norm": float(torch.as_tensor(grad_norm).detach().item()),
        }

    def state_dict(self) -> Dict[str, Any]:
        return {
            "policy": _cpu_state_dict(self.policy),
            "value": _cpu_state_dict(self.value),
        }

    def load_state_dict(self, state: Mapping[str, Any]) -> None:
        self.policy.load_state_dict(state["policy"])
        self.value.load_state_dict(state["value"])


In [ ]:
def _evaluate_maml_checkpoint(
    agent: MAMLAgent,
    velocities: Sequence[float],
    experiment: ExperimentConfig,
    *,
    support_episodes: int,
    seed_base: int,
) -> Tuple[float, float]:
    first_returns: List[float] = []
    adapted_returns: List[float] = []

    for task_index, velocity in enumerate(velocities):
        env = HalfCheetahVel(
            target_velocity=float(velocity),
            env_id=experiment.env_id,
            horizon=experiment.episode_horizon,
        )
        try:
            support_batches: List[TrajectoryBatch] = []
            task_first_return: Optional[float] = None
            for support_index in range(support_episodes):
                support, _, support_return = collect_episode(
                    env,
                    agent,
                    deterministic=False,
                    device=agent.device,
                    reset_seed=seed_base + 100 * task_index + support_index,
                )
                agent.prepare_batch(support)
                support_batches.append(support)
                if task_first_return is None:
                    task_first_return = support_return

            support_batch = _concat_trajectory_batches(support_batches)
            fast_params = agent.adapt(
                support_batch,
                track_meta_gradient=False,
            )

            query, _, query_return = collect_episode(
                env,
                agent,
                deterministic=False,
                device=agent.device,
                agent_state=fast_params,
                reset_seed=seed_base + 100 * task_index + support_episodes,
            )
            first_returns.append(float(task_first_return))
            adapted_returns.append(float(query_return))
        finally:
            env.close()

    return float(np.mean(first_returns)), float(np.mean(adapted_returns))


def evaluate_maml_rollouts_from_history(
    history: Mapping[str, Any],
    experiment: ExperimentConfig,
    velocities: Sequence[float],
    *,
    num_rollouts: int,
) -> np.ndarray:
    """
    Figure-4 diagnostic for vanilla MAML.

    Rollout 1 uses the learned meta-initialization.  After each rollout, that
    rollout is treated as a support set and the declared inner update is applied
    to the *current* fast parameters.  The resulting fast parameters are kept
    for the next rollout and reset only when the task changes.
    """
    obs_dim, action_dim, action_scale = _environment_dimensions(
        experiment, float(velocities[0])
    )
    agent_cfg = dict(history["metadata"]["agent_config"])
    agent_cfg["action_scale"] = action_scale
    agent_cfg["device"] = DEVICE
    agent = MAMLAgent(obs_dim, action_dim, agent_cfg)
    agent.load_state_dict(history["checkpoint"])

    result = np.empty((len(velocities), num_rollouts), dtype=np.float64)
    seed_base = 500_000 + int(history["seed"]) * 10_000

    for task_index, velocity in enumerate(velocities):
        env = HalfCheetahVel(
            target_velocity=float(velocity),
            env_id=experiment.env_id,
            horizon=experiment.episode_horizon,
        )
        fast_params: Optional[Mapping[str, torch.Tensor]] = None
        try:
            for rollout_index in range(num_rollouts):
                batch, _, episode_return = collect_episode(
                    env,
                    agent,
                    deterministic=False,
                    device=agent.device,
                    agent_state=fast_params,
                    reset_seed=seed_base + task_index * 100 + rollout_index,
                )
                result[task_index, rollout_index] = episode_return
                agent.prepare_batch(batch)
                fast_params = agent.adapt(
                    batch,
                    initial_params=fast_params,
                    track_meta_gradient=False,
                )
        finally:
            env.close()

    return result


def train_maml(
    config: ExperimentConfig,
    task_splits: TaskSplits,
    seed: int,
) -> Dict[str, Any]:
    """Meta-train vanilla MAML using training tasks only."""
    seed_everything(seed)
    rng = np.random.default_rng(seed)

    obs_dim, action_dim, action_scale = _environment_dimensions(
        config, float(task_splits.train[0])
    )
    agent_config = _shared_agent_config(config, MAML_CONFIG)
    agent_config["action_scale"] = action_scale
    agent = MAMLAgent(
        obs_dim,
        action_dim,
        {**agent_config, "device": DEVICE},
    )

    meta_batch_size = int(agent_config["meta_batch_size"])
    support_episodes = int(agent_config["support_episodes"])
    query_episodes = int(agent_config["query_episodes"])

    frames = 0
    next_evaluation = int(config.evaluation_interval_frames)
    history_frames: List[int] = []
    first_history: List[float] = []
    adapted_history: List[float] = []

    update_index = 0
    progress = tqdm(
        total=config.total_frames,
        desc=f"MAML seed {seed}",
        unit="frame",
        leave=False,
    )

    while frames < config.total_frames:
        sampled_velocities = rng.choice(
            task_splits.train,
            size=meta_batch_size,
            replace=len(task_splits.train) < meta_batch_size,
        )
        task_data: List[Dict[str, TrajectoryBatch]] = []
        frames_before = frames

        for task_offset, velocity in enumerate(sampled_velocities):
            env = HalfCheetahVel(
                target_velocity=float(velocity),
                env_id=config.env_id,
                horizon=config.episode_horizon,
            )
            try:
                support_batches: List[TrajectoryBatch] = []
                for support_index in range(support_episodes):
                    batch, _, _ = collect_episode(
                        env,
                        agent,
                        deterministic=False,
                        device=agent.device,
                        reset_seed=int(
                            rng.integers(0, np.iinfo(np.int32).max)
                        ),
                    )
                    agent.prepare_batch(batch)
                    support_batches.append(batch)
                    frames += int(batch.rewards.numel())

                support = _concat_trajectory_batches(support_batches)

                # A numerically identical detached fast update is used only to
                # collect on-policy query data.  meta_update recomputes the
                # differentiable fast update from the same support batch.
                fast_for_collection = agent.adapt(
                    support,
                    track_meta_gradient=False,
                )

                query_batches: List[TrajectoryBatch] = []
                for query_index in range(query_episodes):
                    batch, _, _ = collect_episode(
                        env,
                        agent,
                        deterministic=False,
                        device=agent.device,
                        agent_state=fast_for_collection,
                        reset_seed=int(
                            rng.integers(0, np.iinfo(np.int32).max)
                        ),
                    )
                    agent.prepare_batch(batch)
                    query_batches.append(batch)
                    frames += int(batch.rewards.numel())

                query = _concat_trajectory_batches(query_batches)
                task_data.append({"support": support, "query": query})
            finally:
                env.close()

        metrics = agent.meta_update(task_data)
        update_index += 1
        progress.update(frames - frames_before)

        if (
            update_index % max(1, config.verbose_interval_updates) == 0
        ):
            progress.set_postfix(
                outer=f"{metrics['outer_loss']:.3f}",
                value=f"{metrics['value_loss']:.3f}",
            )

        should_evaluate = frames >= next_evaluation
        if frames >= config.total_frames:
            should_evaluate = True

        if should_evaluate:
            eval_velocities = task_splits.validation[
                : min(config.evaluation_tasks, len(task_splits.validation))
            ]
            first_return, adapted_return = _evaluate_maml_checkpoint(
                agent,
                eval_velocities,
                config,
                support_episodes=support_episodes,
                seed_base=100_000 + seed * 10_000,
            )
            if not history_frames or frames > history_frames[-1]:
                history_frames.append(frames)
                first_history.append(first_return)
                adapted_history.append(adapted_return)

            while next_evaluation <= frames:
                next_evaluation += int(config.evaluation_interval_frames)

    progress.close()

    metadata_agent_config = dict(agent_config)
    metadata_agent_config["action_scale"] = np.asarray(action_scale).tolist()

    history = {
        "method": "MAML",
        "seed": int(seed),
        "frames": np.asarray(history_frames, dtype=np.float64),
        "first_rollout_return": np.asarray(first_history, dtype=np.float64),
        "adapted_return": np.asarray(adapted_history, dtype=np.float64),
        "metadata": {
            "inner_lr": agent.inner_lr,
            "inner_steps": agent.inner_steps,
            "support_episodes": support_episodes,
            "query_episodes": query_episodes,
            "meta_batch_size": meta_batch_size,
            "exact_or_first_order": (
                "first_order" if agent.first_order else "exact"
            ),
            "outer_optimizer": "Adam",
            "policy_architecture": list(
                agent_config.get("hidden_sizes", (64, 64))
            ),
            "evaluation_split": "validation",
            "test_tasks_used_for_gradients": False,
            "agent_config": metadata_agent_config,
        },
        "checkpoint": agent.state_dict(),
    }
    validate_history(history)
    return history


# Part III - RL2

## Algorithm summary

RL2 represents the fast learning algorithm in the hidden state of a recurrent neural network. The
network weights are learned slowly across tasks by an ordinary RL optimizer.

At time $t$, the recurrent policy typically receives:

$$
x_t = (s_t, a_{t-1}, r_t, d_{t-1}),
$$

where $d_{t-1}$ indicates an episode boundary. The recurrent state carries information about the
unknown task.

### Required state-reset semantics

- Reset the recurrent state at the beginning of a **new task**.
- Do **not** reset it between the first and second episodes of the same task.
- Reset the physical HalfCheetah state between episodes.
- Feed a done/reset signal to the recurrent network.
- Preserve temporal order during optimization.
- The target velocity must remain hidden.

### Training objective

Treat the two-episode sequence as one meta-RL trial of length $H^+=400$. Optimize the recurrent
policy to maximize the total return over the trial. PPO or TRPO may be used if implemented
correctly and documented.

In [ ]:
# RL2 uses a GRU.  The GRU state is the task-adaptation state: it is reset only
# when the target-velocity task changes, never at the physical reset between
# rollouts of the same task.


In [ ]:
class RL2Policy(nn.Module):
    """
    Recurrent Gaussian actor-critic for RL2.

    Sequence convention: ``inputs`` has shape [T, B, D], where
    D = observation_dim + action_dim + 2 for previous reward and previous
    done/reset flag.  The recurrent state has shape [1, B, hidden_size].
    """

    def __init__(self, observation_dim: int, action_dim: int, **kwargs):
        super().__init__()
        self.observation_dim = int(observation_dim)
        self.action_dim = int(action_dim)
        self.hidden_size = int(kwargs.get("hidden_size", 128))
        self.input_dim = self.observation_dim + self.action_dim + 2

        action_scale = np.asarray(
            kwargs.get("action_scale", np.ones(action_dim, dtype=np.float32)),
            dtype=np.float32,
        )
        self.register_buffer(
            "action_scale",
            torch.as_tensor(action_scale, dtype=torch.float32),
        )

        self.input_encoder = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_size),
            nn.Tanh(),
        )
        self.gru = nn.GRU(
            input_size=self.hidden_size,
            hidden_size=self.hidden_size,
            num_layers=1,
            batch_first=False,
        )
        self.actor_mean = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Tanh(),
            nn.Linear(self.hidden_size, self.action_dim),
        )
        self.value_head = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.Tanh(),
            nn.Linear(self.hidden_size, 1),
        )
        self.log_std = nn.Parameter(
            torch.full(
                (self.action_dim,),
                float(kwargs.get("init_log_std", -0.5)),
                dtype=torch.float32,
            )
        )

    def initial_state(self, batch_size: int, device: torch.device) -> torch.Tensor:
        return torch.zeros(
            1, int(batch_size), self.hidden_size, device=device
        )

    def forward(
        self,
        inputs: torch.Tensor,
        recurrent_state: Any,
    ) -> Tuple[SquashedDiagGaussian, torch.Tensor, torch.Tensor]:
        """
        inputs: [T, B, D] (or [B, D] for one recurrent step).
        values: [T, B].
        """
        squeeze_time = False
        if inputs.ndim == 2:
            inputs = inputs.unsqueeze(0)
            squeeze_time = True
        if inputs.ndim != 3 or inputs.shape[-1] != self.input_dim:
            raise ValueError(
                f"RL2 inputs must be [T,B,{self.input_dim}], got {tuple(inputs.shape)}."
            )

        encoded = self.input_encoder(inputs)
        output, next_state = self.gru(encoded, recurrent_state)
        mean = self.actor_mean(output)
        log_std = self.log_std.clamp(-5.0, 2.0).view(
            1, 1, -1
        ).expand_as(mean)
        dist = SquashedDiagGaussian(mean, log_std, self.action_scale)
        values = self.value_head(output).squeeze(-1)

        if squeeze_time:
            # Keep batch dimension, remove only the synthetic time dimension.
            dist = SquashedDiagGaussian(
                dist.mean.squeeze(0),
                dist.log_std.squeeze(0),
                self.action_scale,
            )
            values = values.squeeze(0)
        return dist, values, next_state

    def act(
        self,
        observation: np.ndarray,
        previous_action: np.ndarray,
        previous_reward: float,
        previous_done: bool,
        recurrent_state: Any,
        deterministic: bool = False,
    ) -> Tuple[np.ndarray, float, float, torch.Tensor]:
        device = next(self.parameters()).device
        input_vector = np.concatenate(
            [
                np.asarray(observation, dtype=np.float32).reshape(-1),
                np.asarray(previous_action, dtype=np.float32).reshape(-1),
                np.asarray([previous_reward], dtype=np.float32),
                np.asarray([float(previous_done)], dtype=np.float32),
            ]
        )
        inputs = torch.as_tensor(
            input_vector, dtype=torch.float32, device=device
        ).view(1, -1)

        with torch.no_grad():
            dist, value, next_state = self.forward(inputs, recurrent_state)
            action = dist.mode if deterministic else dist.sample()
            log_prob = dist.log_prob(action)

        return (
            action.squeeze(0).cpu().numpy().astype(np.float32),
            float(log_prob.squeeze(0).item()),
            float(value.squeeze(0).item()),
            next_state.detach(),
        )


In [ ]:
class RL2Agent:
    """RL2 with PPO updates over complete, ordered recurrent sequences."""

    def __init__(self, observation_dim: int, action_dim: int, config: Any):
        self.config = dict(config) if isinstance(config, Mapping) else vars(config)
        self.device = torch.device(_cfg_get(config, "device", DEVICE))
        self.observation_dim = int(observation_dim)
        self.action_dim = int(action_dim)

        self.gamma = float(_cfg_get(config, "gamma", 0.97))
        self.gae_lambda = float(_cfg_get(config, "gae_lambda", 0.90))
        self.clip_ratio = float(_cfg_get(config, "clip_ratio", 0.2))
        self.ppo_epochs = int(_cfg_get(config, "ppo_epochs", 4))
        self.value_coef = float(_cfg_get(config, "value_coef", 0.5))
        self.entropy_coef = float(_cfg_get(config, "entropy_coef", 1e-3))
        self.max_grad_norm = float(_cfg_get(config, "max_grad_norm", 0.5))
        self.sequence_minibatch_size = int(
            _cfg_get(config, "sequence_minibatch_size", 4)
        )
        self.episode_horizon = int(_cfg_get(config, "episode_horizon", 200))
        self.env_id = str(_cfg_get(config, "env_id", "HalfCheetah-v5"))

        self.policy = RL2Policy(
            observation_dim,
            action_dim,
            hidden_size=int(_cfg_get(config, "hidden_size", 128)),
            action_scale=_cfg_get(
                config, "action_scale", np.ones(action_dim, dtype=np.float32)
            ),
            init_log_std=float(_cfg_get(config, "init_log_std", -0.5)),
        ).to(self.device)
        self.optimizer = optim.Adam(
            self.policy.parameters(),
            lr=float(_cfg_get(config, "learning_rate", 3e-4)),
        )

    def collect_trial(
        self,
        task_velocity: float,
        *,
        num_episodes: int = 2,
        deterministic: bool = False,
        seed: Optional[int] = None,
    ) -> Dict[str, Any]:
        """
        Collect ordered rollouts from one task.

        The physical environment is reset each episode, while the GRU state is
        retained.  ``previous_done=True`` is supplied on the first step after
        every physical reset (including the initial task reset).
        """
        env = HalfCheetahVel(
            target_velocity=float(task_velocity),
            env_id=self.env_id,
            horizon=self.episode_horizon,
        )
        recurrent_state = self.policy.initial_state(1, self.device)
        previous_action = np.zeros(self.action_dim, dtype=np.float32)
        previous_reward = 0.0
        previous_done = True

        inputs: List[np.ndarray] = []
        observations: List[np.ndarray] = []
        actions: List[np.ndarray] = []
        rewards: List[float] = []
        next_observations: List[np.ndarray] = []
        terminated_flags: List[bool] = []
        truncated_flags: List[bool] = []
        done_flags: List[bool] = []
        log_probs: List[float] = []
        values: List[float] = []
        episode_ids: List[int] = []
        episode_returns: List[float] = []

        try:
            for episode_index in range(int(num_episodes)):
                observation, _ = env.reset(
                    seed=None if seed is None else seed + episode_index
                )
                episode_return = 0.0

                for _ in range(self.episode_horizon):
                    input_vector = np.concatenate(
                        [
                            np.asarray(observation, dtype=np.float32).reshape(-1),
                            previous_action.reshape(-1),
                            np.asarray([previous_reward], dtype=np.float32),
                            np.asarray([float(previous_done)], dtype=np.float32),
                        ]
                    )

                    (
                        action,
                        log_prob,
                        value,
                        recurrent_state,
                    ) = self.policy.act(
                        observation,
                        previous_action,
                        previous_reward,
                        previous_done,
                        recurrent_state,
                        deterministic=deterministic,
                    )

                    (
                        next_observation,
                        reward,
                        terminated,
                        truncated,
                        _,
                    ) = env.step(action)
                    done = bool(terminated or truncated)

                    inputs.append(input_vector)
                    observations.append(
                        np.asarray(observation, dtype=np.float32)
                    )
                    actions.append(action)
                    rewards.append(float(reward))
                    next_observations.append(
                        np.asarray(next_observation, dtype=np.float32)
                    )
                    terminated_flags.append(bool(terminated))
                    truncated_flags.append(bool(truncated))
                    done_flags.append(done)
                    log_probs.append(log_prob)
                    values.append(value)
                    episode_ids.append(episode_index)

                    episode_return += float(reward)
                    previous_action = action
                    previous_reward = float(reward)
                    previous_done = done
                    observation = next_observation

                    if done:
                        break

                episode_returns.append(float(episode_return))
                # Do not reset recurrent_state, previous action/reward/done.
                # The next input therefore communicates the boundary to the GRU.
        finally:
            env.close()

        return {
            "inputs": torch.as_tensor(
                np.asarray(inputs), dtype=torch.float32, device=self.device
            ),
            "observations": torch.as_tensor(
                np.asarray(observations), dtype=torch.float32, device=self.device
            ),
            "actions": torch.as_tensor(
                np.asarray(actions), dtype=torch.float32, device=self.device
            ),
            "rewards": torch.as_tensor(
                rewards, dtype=torch.float32, device=self.device
            ),
            "next_observations": torch.as_tensor(
                np.asarray(next_observations),
                dtype=torch.float32,
                device=self.device,
            ),
            "terminated": torch.as_tensor(
                terminated_flags, dtype=torch.bool, device=self.device
            ),
            "truncated": torch.as_tensor(
                truncated_flags, dtype=torch.bool, device=self.device
            ),
            "dones_for_agent": torch.as_tensor(
                done_flags, dtype=torch.bool, device=self.device
            ),
            "log_probs": torch.as_tensor(
                log_probs, dtype=torch.float32, device=self.device
            ),
            "values": torch.as_tensor(
                values, dtype=torch.float32, device=self.device
            ),
            "episode_ids": torch.as_tensor(
                episode_ids, dtype=torch.long, device=self.device
            ),
            "episode_returns": np.asarray(
                episode_returns, dtype=np.float64
            ),
        }

    def update(self, ordered_trials: Sequence[Any]) -> Dict[str, float]:
        """
        PPO over full recurrent sequences.

        We may shuffle the *task/trial* dimension between PPO minibatches, but
        never individual timesteps.  Every recurrent replay begins from the
        correct zero state for the start of that task.
        """
        if not ordered_trials:
            raise ValueError("RL2 update requires at least one trial.")

        lengths = [int(trial["rewards"].shape[0]) for trial in ordered_trials]
        if len(set(lengths)) != 1:
            raise ValueError(
                "RL2 PPO currently expects equal-length complete trials; "
                f"got lengths {lengths}."
            )

        T = lengths[0]
        B = len(ordered_trials)
        inputs = torch.stack(
            [trial["inputs"] for trial in ordered_trials], dim=1
        )  # [T,B,D]
        actions = torch.stack(
            [trial["actions"] for trial in ordered_trials], dim=1
        )  # [T,B,A]
        old_log_probs = torch.stack(
            [trial["log_probs"] for trial in ordered_trials], dim=1
        ).detach()
        old_values = torch.stack(
            [trial["values"] for trial in ordered_trials], dim=1
        ).detach()

        returns_list: List[torch.Tensor] = []
        advantages_list: List[torch.Tensor] = []
        for trial in ordered_trials:
            returns, advantages = _meta_trial_returns_and_advantages(
                trial["rewards"],
                trial["values"],
                self.gamma,
                self.gae_lambda,
            )
            returns_list.append(returns)
            advantages_list.append(advantages)

        returns = torch.stack(returns_list, dim=1).detach()
        advantages = torch.stack(advantages_list, dim=1).detach()
        advantages = (
            advantages - advantages.mean()
        ) / (advantages.std(unbiased=False) + 1e-8)

        minibatch_size = min(self.sequence_minibatch_size, B)
        last_policy_loss = torch.zeros((), device=self.device)
        last_value_loss = torch.zeros((), device=self.device)
        last_entropy = torch.zeros((), device=self.device)
        last_approx_kl = torch.zeros((), device=self.device)

        for _ in range(self.ppo_epochs):
            permutation = torch.randperm(B, device=self.device)
            for start in range(0, B, minibatch_size):
                index = permutation[start : start + minibatch_size]

                mb_inputs = inputs[:, index]
                mb_actions = actions[:, index]
                mb_old_log_probs = old_log_probs[:, index]
                mb_returns = returns[:, index]
                mb_advantages = advantages[:, index]

                initial_state = self.policy.initial_state(
                    len(index), self.device
                )
                dist, value_pred, _ = self.policy(
                    mb_inputs, initial_state
                )
                new_log_probs = dist.log_prob(mb_actions)
                ratio = (new_log_probs - mb_old_log_probs).exp()

                unclipped = ratio * mb_advantages
                clipped = torch.clamp(
                    ratio,
                    1.0 - self.clip_ratio,
                    1.0 + self.clip_ratio,
                ) * mb_advantages
                policy_loss = -torch.minimum(
                    unclipped, clipped
                ).mean()
                value_loss = F.mse_loss(value_pred, mb_returns)
                entropy = dist.entropy().mean()

                loss = (
                    policy_loss
                    + self.value_coef * value_loss
                    - self.entropy_coef * entropy
                )

                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(
                    self.policy.parameters(), self.max_grad_norm
                )
                self.optimizer.step()

                with torch.no_grad():
                    approx_kl = (
                        mb_old_log_probs - new_log_probs
                    ).mean()

                last_policy_loss = policy_loss.detach()
                last_value_loss = value_loss.detach()
                last_entropy = entropy.detach()
                last_approx_kl = approx_kl.detach()

        return {
            "policy_loss": float(last_policy_loss.item()),
            "value_loss": float(last_value_loss.item()),
            "entropy": float(last_entropy.item()),
            "approx_kl": float(last_approx_kl.item()),
        }

    def evaluate_task(
        self,
        task_velocity: float,
        *,
        num_episodes: int = 2,
        deterministic: bool = False,
        seed: Optional[int] = None,
    ) -> np.ndarray:
        trial = self.collect_trial(
            task_velocity,
            num_episodes=num_episodes,
            deterministic=deterministic,
            seed=seed,
        )
        return np.asarray(trial["episode_returns"], dtype=np.float64)

    def state_dict(self) -> Dict[str, Any]:
        return {"policy": _cpu_state_dict(self.policy)}

    def load_state_dict(self, state: Mapping[str, Any]) -> None:
        self.policy.load_state_dict(state["policy"])


In [ ]:
def _evaluate_rl2_checkpoint(
    agent: RL2Agent,
    velocities: Sequence[float],
    *,
    seed_base: int,
) -> Tuple[float, float]:
    first: List[float] = []
    second: List[float] = []
    for task_index, velocity in enumerate(velocities):
        returns = agent.evaluate_task(
            float(velocity),
            num_episodes=2,
            deterministic=False,
            seed=seed_base + task_index * 100,
        )
        first.append(float(returns[0]))
        second.append(float(returns[1]))
    return float(np.mean(first)), float(np.mean(second))


def evaluate_rl2_rollouts_from_history(
    history: Mapping[str, Any],
    experiment: ExperimentConfig,
    velocities: Sequence[float],
    *,
    num_rollouts: int,
) -> np.ndarray:
    obs_dim, action_dim, action_scale = _environment_dimensions(
        experiment, float(velocities[0])
    )
    agent_cfg = dict(history["metadata"]["agent_config"])
    agent_cfg["action_scale"] = action_scale
    agent_cfg["device"] = DEVICE
    agent = RL2Agent(obs_dim, action_dim, agent_cfg)
    agent.load_state_dict(history["checkpoint"])

    result = np.empty((len(velocities), num_rollouts), dtype=np.float64)
    seed_base = 600_000 + int(history["seed"]) * 10_000
    for task_index, velocity in enumerate(velocities):
        result[task_index] = agent.evaluate_task(
            float(velocity),
            num_episodes=num_rollouts,
            deterministic=False,
            seed=seed_base + task_index * 100,
        )
    return result


def train_rl2(
    config: ExperimentConfig,
    task_splits: TaskSplits,
    seed: int,
) -> Dict[str, Any]:
    """Meta-train RL2 with PPO over ordered two-rollout task trials."""
    seed_everything(seed)
    rng = np.random.default_rng(seed)

    obs_dim, action_dim, action_scale = _environment_dimensions(
        config, float(task_splits.train[0])
    )
    agent_config = _shared_agent_config(config, RL2_CONFIG)
    agent_config["action_scale"] = action_scale
    agent = RL2Agent(
        obs_dim,
        action_dim,
        {**agent_config, "device": DEVICE},
    )

    tasks_per_update = int(agent_config["tasks_per_update"])
    frames = 0
    next_evaluation = int(config.evaluation_interval_frames)
    history_frames: List[int] = []
    first_history: List[float] = []
    adapted_history: List[float] = []
    update_index = 0

    progress = tqdm(
        total=config.total_frames,
        desc=f"RL2 seed {seed}",
        unit="frame",
        leave=False,
    )

    while frames < config.total_frames:
        sampled_velocities = rng.choice(
            task_splits.train,
            size=tasks_per_update,
            replace=len(task_splits.train) < tasks_per_update,
        )
        trials: List[Dict[str, Any]] = []
        frames_before = frames

        for velocity in sampled_velocities:
            trial = agent.collect_trial(
                float(velocity),
                num_episodes=config.recurrent_rollouts_per_task,
                deterministic=False,
                seed=int(rng.integers(0, np.iinfo(np.int32).max)),
            )
            trials.append(trial)
            frames += int(trial["rewards"].numel())

        metrics = agent.update(trials)
        update_index += 1
        progress.update(frames - frames_before)

        if update_index % max(1, config.verbose_interval_updates) == 0:
            progress.set_postfix(
                policy=f"{metrics['policy_loss']:.3f}",
                value=f"{metrics['value_loss']:.3f}",
            )

        should_evaluate = frames >= next_evaluation
        if frames >= config.total_frames:
            should_evaluate = True

        if should_evaluate:
            eval_velocities = task_splits.validation[
                : min(config.evaluation_tasks, len(task_splits.validation))
            ]
            first_return, adapted_return = _evaluate_rl2_checkpoint(
                agent,
                eval_velocities,
                seed_base=100_000 + seed * 10_000,
            )
            if not history_frames or frames > history_frames[-1]:
                history_frames.append(frames)
                first_history.append(first_return)
                adapted_history.append(adapted_return)
            while next_evaluation <= frames:
                next_evaluation += int(config.evaluation_interval_frames)

    progress.close()

    metadata_agent_config = dict(agent_config)
    metadata_agent_config["action_scale"] = np.asarray(action_scale).tolist()

    history = {
        "method": "RL2",
        "seed": int(seed),
        "frames": np.asarray(history_frames, dtype=np.float64),
        "first_rollout_return": np.asarray(first_history, dtype=np.float64),
        "adapted_return": np.asarray(adapted_history, dtype=np.float64),
        "metadata": {
            "recurrent_cell": "GRU",
            "hidden_size": int(agent_config["hidden_size"]),
            "trial_length": int(
                config.recurrent_rollouts_per_task
                * config.episode_horizon
            ),
            "optimizer": "Adam + PPO",
            "ppo_epochs": int(agent_config["ppo_epochs"]),
            "clip_ratio": float(agent_config["clip_ratio"]),
            "number_of_parallel_tasks": int(tasks_per_update),
            "sequence_minibatching_strategy": (
                "shuffle complete task trials only; replay each sequence "
                "from its task-initial zero GRU state"
            ),
            "evaluation_split": "validation",
            "test_tasks_used_for_gradients": False,
            "agent_config": metadata_agent_config,
        },
        "checkpoint": agent.state_dict(),
    }
    validate_history(history)
    return history


# Part IV - VariBAD

## Algorithm summary

VariBAD casts meta-RL as approximate inference in a Bayes-adaptive MDP.

The hidden task or MDP is represented by a latent variable $m$. A recurrent inference network
maintains an approximate posterior

$$
q_\phi(m\mid \tau_{:t}),
$$

where $\tau_{:t}$ is the interaction history. A policy acts on the augmented state

$$
(s_t, q_\phi(m\mid\tau_{:t})).
$$

A decoder is trained with a variational objective so that the latent posterior captures task-relevant
information. In HalfCheetahVel, the published MuJoCo experiments use a **reward decoder**.

A simplified per-time-step ELBO has the form

$$
\mathcal L_{\text{ELBO},t}
=
\mathbb E_{q_\phi(m\mid\tau_{:t})}
[\log p_\theta(r\mid s,a,m)]
-
D_{\mathrm{KL}}
\left(
q_\phi(m\mid\tau_{:t})
\Vert
p(m\mid\tau_{:t-1})
\right).
$$

The policy is trained on the Bayes-adaptive trajectory while conditioning on the inferred posterior.

### Required components

- recurrent encoder of state, action, and reward history;
- posterior mean and variance;
- reparameterized latent sampling;
- reward decoder;
- KL term and reconstruction term;
- latent-conditioned actor and value function;
- on-policy RL update;
- VAE replay/trajectory storage or an equivalent justified implementation;
- posterior retained across the two episodes of a task.

### Evaluation

- Reset the posterior to its prior at the beginning of a new test task.
- Update the posterior online after transitions.
- Preserve the posterior across the environment reset.
- `first_rollout_return`: episode 1.
- `adapted_return`: episode 2 with the posterior retained.

In [ ]:
# VariBAD uses Normal posteriors with diagonal covariance.  ``kl_divergence``
# and reparameterized Normal samples are imported in the shared import cell.


In [ ]:
class VariBADEncoder(nn.Module):
    """
    Recurrent diagonal-Gaussian posterior q_phi(m | trajectory prefix).

    The encoder consumes (state, action, reward, done/reset) after every
    transition.  Its GRU hidden state and posterior are preserved across the
    physical reset between rollouts of the same task.
    """

    def __init__(
        self,
        observation_dim: int,
        action_dim: int,
        latent_dim: int,
        **kwargs,
    ):
        super().__init__()
        self.observation_dim = int(observation_dim)
        self.action_dim = int(action_dim)
        self.latent_dim = int(latent_dim)
        self.hidden_size = int(kwargs.get("hidden_size", 128))
        self.input_dim = self.observation_dim + self.action_dim + 2

        self.input_encoder = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_size),
            nn.Tanh(),
        )
        self.gru_cell = nn.GRUCell(self.hidden_size, self.hidden_size)
        self.posterior_mean = nn.Linear(self.hidden_size, self.latent_dim)
        self.posterior_logvar = nn.Linear(self.hidden_size, self.latent_dim)

    @staticmethod
    def _sample(
        mean: torch.Tensor,
        logvar: torch.Tensor,
    ) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        return mean + std * torch.randn_like(std)

    def prior(
        self,
        batch_size: int,
        device: torch.device,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return standard-Normal prior parameters/sample and zero GRU state."""
        mean = torch.zeros(
            int(batch_size), self.latent_dim, device=device
        )
        logvar = torch.zeros_like(mean)
        latent = self._sample(mean, logvar)
        recurrent_state = torch.zeros(
            int(batch_size), self.hidden_size, device=device
        )
        return mean, logvar, latent, recurrent_state

    def forward(
        self,
        observation: torch.Tensor,
        action: torch.Tensor,
        reward: torch.Tensor,
        recurrent_state: Any,
        done: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        One posterior update.

        Inputs use batch-major shapes [B, ...].  reward and done may be [B] or
        [B,1].  Returns (mean, logvar, reparameterized latent, next GRU state).
        """
        if observation.ndim == 1:
            observation = observation.unsqueeze(0)
        if action.ndim == 1:
            action = action.unsqueeze(0)
        if reward.ndim == 1:
            reward = reward.unsqueeze(-1)
        if done is None:
            done = torch.zeros_like(reward)
        elif done.ndim == 1:
            done = done.unsqueeze(-1)

        encoder_input = torch.cat(
            [observation, action, reward, done.to(reward.dtype)], dim=-1
        )
        embedded = self.input_encoder(encoder_input)
        next_state = self.gru_cell(embedded, recurrent_state)
        mean = self.posterior_mean(next_state)
        logvar = self.posterior_logvar(next_state).clamp(-10.0, 5.0)
        latent = self._sample(mean, logvar)
        return mean, logvar, latent, next_state


In [ ]:
class VariBADRewardDecoder(nn.Module):
    """Reward decoder p(r_t | s_t, a_t, m), matching the notebook ELBO."""

    def __init__(
        self,
        observation_dim: int,
        action_dim: int,
        latent_dim: int,
        **kwargs,
    ):
        super().__init__()
        hidden_sizes = tuple(kwargs.get("hidden_sizes", (64, 32)))
        input_dim = int(observation_dim) + int(action_dim) + int(latent_dim)
        self.network = _mlp(
            input_dim,
            hidden_sizes,
            1,
            activation=nn.ReLU,
        )

    def forward(
        self,
        previous_observation: torch.Tensor,
        action: torch.Tensor,
        next_observation: torch.Tensor,
        latent: torch.Tensor,
    ) -> torch.Tensor:
        # ``next_observation`` remains in the starter-notebook interface, but the
        # stated HalfCheetahVel ELBO conditions the reward decoder on (s_t, a_t, m).
        _ = next_observation
        inputs = torch.cat(
            [previous_observation, action, latent],
            dim=-1,
        )
        return self.network(inputs).squeeze(-1)


In [ ]:
class VariBADPolicy(nn.Module):
    """Latent-conditioned tanh-Gaussian actor with a shared value function."""

    def __init__(
        self,
        observation_dim: int,
        action_dim: int,
        latent_dim: int,
        **kwargs,
    ):
        super().__init__()
        hidden_sizes = tuple(kwargs.get("hidden_sizes", (128, 128)))
        action_scale = np.asarray(
            kwargs.get("action_scale", np.ones(action_dim, dtype=np.float32)),
            dtype=np.float32,
        )
        input_dim = int(observation_dim) + int(latent_dim)

        layers: List[nn.Module] = []
        last = input_dim
        for width in hidden_sizes:
            layers.extend([nn.Linear(last, int(width)), nn.Tanh()])
            last = int(width)
        self.backbone = nn.Sequential(*layers)
        self.actor_mean = nn.Linear(last, int(action_dim))
        self.value_head = nn.Linear(last, 1)
        self.log_std = nn.Parameter(
            torch.full(
                (int(action_dim),),
                float(kwargs.get("init_log_std", -0.5)),
                dtype=torch.float32,
            )
        )
        self.register_buffer(
            "action_scale",
            torch.as_tensor(action_scale, dtype=torch.float32),
        )

    def forward(
        self,
        observation: torch.Tensor,
        latent_belief: torch.Tensor,
    ) -> Tuple[SquashedDiagGaussian, torch.Tensor]:
        features = self.backbone(
            torch.cat([observation, latent_belief], dim=-1)
        )
        mean = self.actor_mean(features)
        log_std = self.log_std.clamp(-5.0, 2.0).expand_as(mean)
        value = self.value_head(features).squeeze(-1)
        return (
            SquashedDiagGaussian(mean, log_std, self.action_scale),
            value,
        )


In [ ]:
class VariBADAgent:
    """
    VariBAD-style latent task inference plus PPO control.

    VAE sign convention:
        vae_loss = reward_reconstruction_MSE + kl_weight * KL

    This is the negative-ELBO convention for minimization.  Reconstruction and
    KL terms are logged separately.  The KL prior at time t is the previous
    posterior; at a new task it is N(0, I).  The recurrent posterior is not
    reset at the physical episode boundary.
    """

    def __init__(self, observation_dim: int, action_dim: int, config: Any):
        self.config = dict(config) if isinstance(config, Mapping) else vars(config)
        self.device = torch.device(_cfg_get(config, "device", DEVICE))
        self.observation_dim = int(observation_dim)
        self.action_dim = int(action_dim)
        self.latent_dim = int(_cfg_get(config, "latent_dim", 5))
        self.episode_horizon = int(_cfg_get(config, "episode_horizon", 200))
        self.env_id = str(_cfg_get(config, "env_id", "HalfCheetah-v5"))

        self.gamma = float(_cfg_get(config, "gamma", 0.97))
        self.gae_lambda = float(_cfg_get(config, "gae_lambda", 0.90))
        self.clip_ratio = float(_cfg_get(config, "clip_ratio", 0.1))
        self.ppo_epochs = int(_cfg_get(config, "ppo_epochs", 4))
        self.value_coef = float(_cfg_get(config, "value_coef", 0.5))
        self.entropy_coef = float(_cfg_get(config, "entropy_coef", 1e-3))
        self.max_grad_norm = float(_cfg_get(config, "max_grad_norm", 0.5))
        self.kl_weight = float(_cfg_get(config, "kl_weight", 0.1))
        self.policy_minibatch_size = int(
            _cfg_get(config, "policy_minibatch_size", 512)
        )
        self.vae_updates = int(_cfg_get(config, "vae_updates", 2))
        self.vae_batch_tasks = int(_cfg_get(config, "vae_batch_tasks", 4))

        self.encoder = VariBADEncoder(
            observation_dim,
            action_dim,
            self.latent_dim,
            hidden_size=int(
                _cfg_get(config, "encoder_hidden_size", 128)
            ),
        ).to(self.device)
        self.decoder = VariBADRewardDecoder(
            observation_dim,
            action_dim,
            self.latent_dim,
            hidden_sizes=_cfg_get(
                config, "decoder_hidden_sizes", (64, 32)
            ),
        ).to(self.device)
        self.policy = VariBADPolicy(
            observation_dim,
            action_dim,
            self.latent_dim,
            hidden_sizes=_cfg_get(
                config, "policy_hidden_sizes", (128, 128)
            ),
            action_scale=_cfg_get(
                config, "action_scale", np.ones(action_dim, dtype=np.float32)
            ),
            init_log_std=float(_cfg_get(config, "init_log_std", -0.5)),
        ).to(self.device)

        self.policy_optimizer = optim.Adam(
            self.policy.parameters(),
            lr=float(
                _cfg_get(config, "policy_learning_rate", 7e-4)
            ),
        )
        self.vae_optimizer = optim.Adam(
            list(self.encoder.parameters()) + list(self.decoder.parameters()),
            lr=float(_cfg_get(config, "vae_learning_rate", 1e-3)),
        )
        self.vae_buffer: deque = deque(
            maxlen=int(_cfg_get(config, "vae_buffer_tasks", 256))
        )

    def reset_belief(
        self,
        batch_size: int = 1,
        *,
        deterministic: bool = False,
    ) -> Dict[str, torch.Tensor]:
        mean, logvar, latent, state = self.encoder.prior(
            batch_size, self.device
        )
        if deterministic:
            latent = mean
        return {
            "mean": mean,
            "logvar": logvar,
            "latent": latent,
            "state": state,
        }

    def update_belief(
        self,
        transition: Mapping[str, Any],
        belief_state: Mapping[str, torch.Tensor],
        *,
        deterministic: bool = False,
        track_grad: bool = False,
    ) -> Dict[str, torch.Tensor]:
        observation = torch.as_tensor(
            transition["observation"],
            dtype=torch.float32,
            device=self.device,
        ).view(1, -1)
        action = torch.as_tensor(
            transition["action"],
            dtype=torch.float32,
            device=self.device,
        ).view(1, -1)
        reward = torch.as_tensor(
            [transition["reward"]],
            dtype=torch.float32,
            device=self.device,
        )
        done = torch.as_tensor(
            [float(transition["done"])],
            dtype=torch.float32,
            device=self.device,
        )

        context = torch.enable_grad() if track_grad else torch.no_grad()
        with context:
            mean, logvar, latent, next_state = self.encoder(
                observation,
                action,
                reward,
                belief_state["state"],
                done,
            )
            if deterministic:
                latent = mean

        if not track_grad:
            mean = mean.detach()
            logvar = logvar.detach()
            latent = latent.detach()
            next_state = next_state.detach()

        return {
            "mean": mean,
            "logvar": logvar,
            "latent": latent,
            "state": next_state,
        }

    def collect_trial(
        self,
        task_velocity: float,
        *,
        num_episodes: int = 2,
        deterministic: bool = False,
        seed: Optional[int] = None,
    ) -> Dict[str, Any]:
        """Collect a Bayes-adaptive task trial while retaining posterior state."""
        env = HalfCheetahVel(
            target_velocity=float(task_velocity),
            env_id=self.env_id,
            horizon=self.episode_horizon,
        )
        belief = self.reset_belief(
            1, deterministic=deterministic
        )

        observations: List[np.ndarray] = []
        actions: List[np.ndarray] = []
        rewards: List[float] = []
        next_observations: List[np.ndarray] = []
        terminated_flags: List[bool] = []
        truncated_flags: List[bool] = []
        done_flags: List[bool] = []
        log_probs: List[float] = []
        values: List[float] = []
        latents: List[np.ndarray] = []
        posterior_means: List[np.ndarray] = []
        posterior_logvars: List[np.ndarray] = []
        episode_ids: List[int] = []
        episode_returns: List[float] = []

        try:
            for episode_index in range(int(num_episodes)):
                observation, _ = env.reset(
                    seed=None if seed is None else seed + episode_index
                )
                episode_return = 0.0

                for _ in range(self.episode_horizon):
                    obs_tensor = torch.as_tensor(
                        observation,
                        dtype=torch.float32,
                        device=self.device,
                    ).view(1, -1)
                    latent_for_action = belief["latent"]

                    with torch.no_grad():
                        dist, value = self.policy(
                            obs_tensor, latent_for_action
                        )
                        action_tensor = (
                            dist.mode if deterministic else dist.sample()
                        )
                        log_prob = dist.log_prob(action_tensor)

                    action = (
                        action_tensor.squeeze(0)
                        .cpu()
                        .numpy()
                        .astype(np.float32)
                    )
                    (
                        next_observation,
                        reward,
                        terminated,
                        truncated,
                        _,
                    ) = env.step(action)
                    done = bool(terminated or truncated)

                    observations.append(
                        np.asarray(observation, dtype=np.float32)
                    )
                    actions.append(action)
                    rewards.append(float(reward))
                    next_observations.append(
                        np.asarray(next_observation, dtype=np.float32)
                    )
                    terminated_flags.append(bool(terminated))
                    truncated_flags.append(bool(truncated))
                    done_flags.append(done)
                    log_probs.append(float(log_prob.item()))
                    values.append(float(value.item()))
                    latents.append(
                        latent_for_action.squeeze(0).detach().cpu().numpy()
                    )
                    posterior_means.append(
                        belief["mean"].squeeze(0).detach().cpu().numpy()
                    )
                    posterior_logvars.append(
                        belief["logvar"].squeeze(0).detach().cpu().numpy()
                    )
                    episode_ids.append(episode_index)

                    belief = self.update_belief(
                        {
                            "observation": observation,
                            "action": action,
                            "reward": float(reward),
                            "done": done,
                        },
                        belief,
                        deterministic=deterministic,
                        track_grad=False,
                    )

                    episode_return += float(reward)
                    observation = next_observation
                    if done:
                        break

                episode_returns.append(float(episode_return))
                # Belief is intentionally retained across this physical reset.
        finally:
            env.close()

        return {
            "observations": torch.as_tensor(
                np.asarray(observations), dtype=torch.float32, device=self.device
            ),
            "actions": torch.as_tensor(
                np.asarray(actions), dtype=torch.float32, device=self.device
            ),
            "rewards": torch.as_tensor(
                rewards, dtype=torch.float32, device=self.device
            ),
            "next_observations": torch.as_tensor(
                np.asarray(next_observations),
                dtype=torch.float32,
                device=self.device,
            ),
            "terminated": torch.as_tensor(
                terminated_flags, dtype=torch.bool, device=self.device
            ),
            "truncated": torch.as_tensor(
                truncated_flags, dtype=torch.bool, device=self.device
            ),
            "dones_for_agent": torch.as_tensor(
                done_flags, dtype=torch.bool, device=self.device
            ),
            "log_probs": torch.as_tensor(
                log_probs, dtype=torch.float32, device=self.device
            ),
            "values": torch.as_tensor(
                values, dtype=torch.float32, device=self.device
            ),
            "latents": torch.as_tensor(
                np.asarray(latents), dtype=torch.float32, device=self.device
            ),
            "posterior_means": torch.as_tensor(
                np.asarray(posterior_means),
                dtype=torch.float32,
                device=self.device,
            ),
            "posterior_logvars": torch.as_tensor(
                np.asarray(posterior_logvars),
                dtype=torch.float32,
                device=self.device,
            ),
            "episode_ids": torch.as_tensor(
                episode_ids, dtype=torch.long, device=self.device
            ),
            "episode_returns": np.asarray(
                episode_returns, dtype=np.float64
            ),
        }

    def compute_elbo(
        self,
        batch: Any,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute the negative ELBO over complete ordered task trials.

        The optimized objective is:
            reconstruction_MSE + kl_weight * KL(q_t || q_{t-1})
        with q_{-1} = N(0, I).
        """
        trials = list(batch) if isinstance(batch, (list, tuple)) else [batch]
        if not trials:
            raise ValueError("VariBAD ELBO requires at least one trial.")

        lengths = [int(trial["rewards"].shape[0]) for trial in trials]
        if len(set(lengths)) != 1:
            raise ValueError(
                "VariBAD VAE replay expects equal-length complete trials."
            )

        observations = torch.stack(
            [
                trial["observations"].to(self.device)
                for trial in trials
            ],
            dim=1,
        )  # [T,B,O]
        actions = torch.stack(
            [trial["actions"].to(self.device) for trial in trials],
            dim=1,
        )
        rewards = torch.stack(
            [trial["rewards"].to(self.device) for trial in trials],
            dim=1,
        )
        next_observations = torch.stack(
            [
                trial["next_observations"].to(self.device)
                for trial in trials
            ],
            dim=1,
        )
        dones = torch.stack(
            [
                trial["dones_for_agent"].to(self.device)
                for trial in trials
            ],
            dim=1,
        )

        T, B = rewards.shape
        prior_mean, prior_logvar, _, recurrent_state = self.encoder.prior(
            B, self.device
        )

        reconstruction_terms: List[torch.Tensor] = []
        kl_terms: List[torch.Tensor] = []

        for t in range(T):
            mean, logvar, latent, recurrent_state = self.encoder(
                observations[t],
                actions[t],
                rewards[t],
                recurrent_state,
                dones[t].to(torch.float32),
            )
            prediction = self.decoder(
                observations[t],
                actions[t],
                next_observations[t],
                latent,
            )
            reconstruction_terms.append(
                F.mse_loss(
                    prediction,
                    rewards[t],
                    reduction="mean",
                )
            )

            posterior = Normal(
                mean, torch.exp(0.5 * logvar)
            )
            prior = Normal(
                prior_mean, torch.exp(0.5 * prior_logvar)
            )
            kl_terms.append(
                kl_divergence(posterior, prior)
                .sum(dim=-1)
                .mean()
            )

            # Sequential prior for the next prefix.  We do not reset at an
            # episode boundary; the done flag has already been encoded.
            prior_mean = mean
            prior_logvar = logvar

        reconstruction_loss = torch.stack(
            reconstruction_terms
        ).mean()
        kl_loss = torch.stack(kl_terms).mean()
        vae_loss = reconstruction_loss + self.kl_weight * kl_loss

        metrics = {
            "reconstruction_loss": float(
                reconstruction_loss.detach().item()
            ),
            "kl_loss": float(kl_loss.detach().item()),
            "vae_loss": float(vae_loss.detach().item()),
        }
        return vae_loss, metrics

    def update_policy(
        self,
        trials: Sequence[Mapping[str, Any]],
    ) -> Dict[str, float]:
        if not trials:
            raise ValueError("VariBAD policy update requires trials.")

        observations_list: List[torch.Tensor] = []
        actions_list: List[torch.Tensor] = []
        latents_list: List[torch.Tensor] = []
        old_log_probs_list: List[torch.Tensor] = []
        returns_list: List[torch.Tensor] = []
        advantages_list: List[torch.Tensor] = []

        for trial in trials:
            returns, advantages = _meta_trial_returns_and_advantages(
                trial["rewards"],
                trial["values"],
                self.gamma,
                self.gae_lambda,
            )
            observations_list.append(trial["observations"])
            actions_list.append(trial["actions"])
            latents_list.append(trial["latents"].detach())
            old_log_probs_list.append(trial["log_probs"].detach())
            returns_list.append(returns.detach())
            advantages_list.append(advantages.detach())

        observations = torch.cat(observations_list, dim=0)
        actions = torch.cat(actions_list, dim=0)
        latents = torch.cat(latents_list, dim=0)
        old_log_probs = torch.cat(old_log_probs_list, dim=0)
        returns = torch.cat(returns_list, dim=0)
        advantages = torch.cat(advantages_list, dim=0)
        advantages = (
            advantages - advantages.mean()
        ) / (advantages.std(unbiased=False) + 1e-8)

        N = observations.shape[0]
        minibatch_size = min(self.policy_minibatch_size, N)
        last_policy_loss = torch.zeros((), device=self.device)
        last_value_loss = torch.zeros((), device=self.device)
        last_entropy = torch.zeros((), device=self.device)

        for _ in range(self.ppo_epochs):
            permutation = torch.randperm(N, device=self.device)
            for start in range(0, N, minibatch_size):
                index = permutation[start : start + minibatch_size]
                dist, value_pred = self.policy(
                    observations[index], latents[index]
                )
                new_log_probs = dist.log_prob(actions[index])
                ratio = (
                    new_log_probs - old_log_probs[index]
                ).exp()
                unclipped = ratio * advantages[index]
                clipped = torch.clamp(
                    ratio,
                    1.0 - self.clip_ratio,
                    1.0 + self.clip_ratio,
                ) * advantages[index]
                policy_loss = -torch.minimum(
                    unclipped, clipped
                ).mean()
                value_loss = F.mse_loss(
                    value_pred, returns[index]
                )
                entropy = dist.entropy().mean()

                loss = (
                    policy_loss
                    + self.value_coef * value_loss
                    - self.entropy_coef * entropy
                )
                self.policy_optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(
                    self.policy.parameters(), self.max_grad_norm
                )
                self.policy_optimizer.step()

                last_policy_loss = policy_loss.detach()
                last_value_loss = value_loss.detach()
                last_entropy = entropy.detach()

        return {
            "policy_loss": float(last_policy_loss.item()),
            "value_loss": float(last_value_loss.item()),
            "entropy": float(last_entropy.item()),
        }

    def update_vae(
        self,
        trials: Sequence[Mapping[str, Any]],
    ) -> Dict[str, float]:
        vae_loss, metrics = self.compute_elbo(trials)
        self.vae_optimizer.zero_grad(set_to_none=True)
        vae_loss.backward()
        nn.utils.clip_grad_norm_(
            list(self.encoder.parameters()) + list(self.decoder.parameters()),
            self.max_grad_norm,
        )
        self.vae_optimizer.step()
        return metrics

    def update(self, batch: Any) -> Dict[str, float]:
        trials = list(batch) if isinstance(batch, (list, tuple)) else [batch]
        policy_metrics = self.update_policy(trials)

        for trial in trials:
            self.vae_buffer.append(_copy_trial_to_cpu(trial))

        vae_metrics_accumulator: List[Dict[str, float]] = []
        for _ in range(self.vae_updates):
            sample_count = min(
                self.vae_batch_tasks, len(self.vae_buffer)
            )
            sampled = random.sample(
                list(self.vae_buffer), k=sample_count
            )
            vae_metrics_accumulator.append(self.update_vae(sampled))

        metrics = dict(policy_metrics)
        if vae_metrics_accumulator:
            for key in (
                "reconstruction_loss",
                "kl_loss",
                "vae_loss",
            ):
                metrics[key] = float(
                    np.mean(
                        [entry[key] for entry in vae_metrics_accumulator]
                    )
                )
        return metrics

    def evaluate_task(
        self,
        task_velocity: float,
        *,
        num_episodes: int = 2,
        deterministic: bool = False,
        seed: Optional[int] = None,
    ) -> np.ndarray:
        trial = self.collect_trial(
            task_velocity,
            num_episodes=num_episodes,
            deterministic=deterministic,
            seed=seed,
        )
        return np.asarray(trial["episode_returns"], dtype=np.float64)

    def state_dict(self) -> Dict[str, Any]:
        return {
            "encoder": _cpu_state_dict(self.encoder),
            "decoder": _cpu_state_dict(self.decoder),
            "policy": _cpu_state_dict(self.policy),
        }

    def load_state_dict(self, state: Mapping[str, Any]) -> None:
        self.encoder.load_state_dict(state["encoder"])
        self.decoder.load_state_dict(state["decoder"])
        self.policy.load_state_dict(state["policy"])


In [ ]:
def _evaluate_varibad_checkpoint(
    agent: VariBADAgent,
    velocities: Sequence[float],
    *,
    seed_base: int,
) -> Tuple[float, float]:
    first: List[float] = []
    second: List[float] = []
    for task_index, velocity in enumerate(velocities):
        returns = agent.evaluate_task(
            float(velocity),
            num_episodes=2,
            deterministic=False,
            seed=seed_base + task_index * 100,
        )
        first.append(float(returns[0]))
        second.append(float(returns[1]))
    return float(np.mean(first)), float(np.mean(second))


def evaluate_varibad_rollouts_from_history(
    history: Mapping[str, Any],
    experiment: ExperimentConfig,
    velocities: Sequence[float],
    *,
    num_rollouts: int,
) -> np.ndarray:
    obs_dim, action_dim, action_scale = _environment_dimensions(
        experiment, float(velocities[0])
    )
    agent_cfg = dict(history["metadata"]["agent_config"])
    agent_cfg["action_scale"] = action_scale
    agent_cfg["device"] = DEVICE
    agent = VariBADAgent(obs_dim, action_dim, agent_cfg)
    agent.load_state_dict(history["checkpoint"])

    result = np.empty((len(velocities), num_rollouts), dtype=np.float64)
    seed_base = 700_000 + int(history["seed"]) * 10_000
    for task_index, velocity in enumerate(velocities):
        result[task_index] = agent.evaluate_task(
            float(velocity),
            num_episodes=num_rollouts,
            deterministic=False,
            seed=seed_base + task_index * 100,
        )
    return result


def train_varibad(
    config: ExperimentConfig,
    task_splits: TaskSplits,
    seed: int,
) -> Dict[str, Any]:
    """Meta-train VariBAD using on-policy PPO plus recurrent VAE replay."""
    seed_everything(seed)
    rng = np.random.default_rng(seed)

    obs_dim, action_dim, action_scale = _environment_dimensions(
        config, float(task_splits.train[0])
    )
    agent_config = _shared_agent_config(config, VARIBAD_CONFIG)
    agent_config["action_scale"] = action_scale
    agent = VariBADAgent(
        obs_dim,
        action_dim,
        {**agent_config, "device": DEVICE},
    )

    tasks_per_update = int(agent_config["tasks_per_update"])
    frames = 0
    next_evaluation = int(config.evaluation_interval_frames)
    history_frames: List[int] = []
    first_history: List[float] = []
    adapted_history: List[float] = []
    update_index = 0

    # Separate VAE diagnostics required by the assignment.  These are training
    # diagnostics only; validation/test tasks never enter a gradient update.
    diagnostic_frames: List[int] = []
    reconstruction_history: List[float] = []
    kl_history: List[float] = []
    vae_loss_history: List[float] = []

    progress = tqdm(
        total=config.total_frames,
        desc=f"VariBAD seed {seed}",
        unit="frame",
        leave=False,
    )

    while frames < config.total_frames:
        sampled_velocities = rng.choice(
            task_splits.train,
            size=tasks_per_update,
            replace=len(task_splits.train) < tasks_per_update,
        )
        trials: List[Dict[str, Any]] = []
        frames_before = frames

        for velocity in sampled_velocities:
            trial = agent.collect_trial(
                float(velocity),
                num_episodes=config.recurrent_rollouts_per_task,
                deterministic=False,
                seed=int(rng.integers(0, np.iinfo(np.int32).max)),
            )
            trials.append(trial)
            frames += int(trial["rewards"].numel())

        metrics = agent.update(trials)
        update_index += 1
        progress.update(frames - frames_before)

        diagnostic_frames.append(int(frames))
        reconstruction_history.append(
            float(metrics.get("reconstruction_loss", float("nan")))
        )
        kl_history.append(float(metrics.get("kl_loss", float("nan"))))
        vae_loss_history.append(float(metrics.get("vae_loss", float("nan"))))

        if update_index % max(1, config.verbose_interval_updates) == 0:
            progress.set_postfix(
                policy=f"{metrics['policy_loss']:.3f}",
                recon=f"{metrics.get('reconstruction_loss', float('nan')):.3f}",
                kl=f"{metrics.get('kl_loss', float('nan')):.3f}",
            )

        should_evaluate = frames >= next_evaluation
        if frames >= config.total_frames:
            should_evaluate = True

        if should_evaluate:
            eval_velocities = task_splits.validation[
                : min(config.evaluation_tasks, len(task_splits.validation))
            ]
            first_return, adapted_return = _evaluate_varibad_checkpoint(
                agent,
                eval_velocities,
                seed_base=100_000 + seed * 10_000,
            )
            if not history_frames or frames > history_frames[-1]:
                history_frames.append(frames)
                first_history.append(first_return)
                adapted_history.append(adapted_return)
            while next_evaluation <= frames:
                next_evaluation += int(config.evaluation_interval_frames)

    progress.close()

    metadata_agent_config = dict(agent_config)
    metadata_agent_config["action_scale"] = np.asarray(action_scale).tolist()

    history = {
        "method": "VariBAD",
        "seed": int(seed),
        "frames": np.asarray(history_frames, dtype=np.float64),
        "first_rollout_return": np.asarray(first_history, dtype=np.float64),
        "adapted_return": np.asarray(adapted_history, dtype=np.float64),
        "training_diagnostics": {
            "frames": np.asarray(diagnostic_frames, dtype=np.float64),
            "reconstruction_loss": np.asarray(
                reconstruction_history, dtype=np.float64
            ),
            "kl_loss": np.asarray(kl_history, dtype=np.float64),
            "vae_loss": np.asarray(vae_loss_history, dtype=np.float64),
        },
        "metadata": {
            "latent_dim": int(agent.latent_dim),
            "encoder_hidden_size": int(
                agent_config["encoder_hidden_size"]
            ),
            "policy_hidden_sizes": list(
                agent_config["policy_hidden_sizes"]
            ),
            "decoder_hidden_sizes": list(
                agent_config["decoder_hidden_sizes"]
            ),
            "kl_weight": float(agent.kl_weight),
            "vae_loss_sign_convention": (
                "minimize reward_MSE + kl_weight * KL"
            ),
            "ppo_clip": float(agent.clip_ratio),
            "policy_learning_rate": float(
                agent_config["policy_learning_rate"]
            ),
            "vae_learning_rate": float(
                agent_config["vae_learning_rate"]
            ),
            "trial_length": int(
                config.recurrent_rollouts_per_task
                * config.episode_horizon
            ),
            "posterior_reset": (
                "new task only; preserved across physical episode reset"
            ),
            "evaluation_split": "validation",
            "test_tasks_used_for_gradients": False,
            "agent_config": metadata_agent_config,
        },
        "checkpoint": agent.state_dict(),
    }
    validate_history(history)
    return history


# Part V - Training

In [ ]:
# Method-specific hyperparameters.
#
# CONFIG is DEBUG_CONFIG by default so the notebook can be smoke-tested in
# Colab.  For the assignment-scale experiment set CONFIG = FULL_CONFIG in the
# configuration cell before creating TASKS and running training.

MAML_CONFIG = {
    "meta_batch_size": 4,
    "support_episodes": 1,
    "query_episodes": 1,
    "inner_steps": 1,
    "inner_lr": 0.10,
    "first_order": False,  # exact/vanilla MAML
    "outer_lr": 3e-4,
    "value_lr": 1e-3,
    "value_epochs": 5,
    "hidden_sizes": (64, 64),
    "value_hidden_sizes": (64, 64),
    "init_log_std": -0.5,
    "entropy_coef": 0.0,
    "outer_optimizer": "Adam",
}

RL2_CONFIG = {
    "hidden_size": 128,
    "tasks_per_update": 4,
    "ppo_epochs": 4,
    "clip_ratio": 0.20,
    "learning_rate": 3e-4,
    "value_coef": 0.5,
    "entropy_coef": 1e-3,
    "sequence_minibatch_size": 4,
    "init_log_std": -0.5,
}

VARIBAD_CONFIG = {
    "latent_dim": 5,
    "encoder_hidden_size": 128,
    "policy_hidden_sizes": (128, 128),
    "decoder_hidden_sizes": (64, 32),
    "kl_weight": 0.1,
    "policy_learning_rate": 7e-4,
    "vae_learning_rate": 1e-3,
    "tasks_per_update": 4,
    "ppo_epochs": 4,
    "clip_ratio": 0.10,
    "value_coef": 0.5,
    "entropy_coef": 1e-3,
    "policy_minibatch_size": 512,
    "vae_updates": 2,
    "vae_batch_tasks": 4,
    "vae_buffer_tasks": 256,
    "init_log_std": -0.5,
}


In [ ]:
# Run every seed requested by CONFIG.  With the notebook default
# CONFIG = DEBUG_CONFIG this is a short interface/training smoke test.
# Switch to FULL_CONFIG (and rebuild TASKS) for the assignment-scale run.

all_histories: Dict[str, List[Dict[str, Any]]] = {
    "VariBAD": [],
    "RL2": [],
    "MAML": [],
}

for seed in CONFIG.seeds:
    print(f"\n=== Seed {seed}: VariBAD ===")
    all_histories["VariBAD"].append(
        train_varibad(CONFIG, TASKS, int(seed))
    )

    print(f"\n=== Seed {seed}: RL2 ===")
    all_histories["RL2"].append(
        train_rl2(CONFIG, TASKS, int(seed))
    )

    print(f"\n=== Seed {seed}: MAML ===")
    all_histories["MAML"].append(
        train_maml(CONFIG, TASKS, int(seed))
    )

for runs in all_histories.values():
    for run in runs:
        validate_history(run)

print("Completed runs:", {k: len(v) for k, v in all_histories.items()})


# Part VI - Provided plotting tools

The plotting code below is complete. Your training functions only need to return the required
history dictionaries.

Curves from different seeds are linearly interpolated onto a shared frame grid before averaging.
The shaded region is a normal-approximation 95% confidence interval:

$$
\bar{x} \pm 1.96\frac{s}{\sqrt{n}}.
$$

With very few seeds, also show individual seed curves and discuss uncertainty.

### Figure 4 aggregation

For the Figure 4-style plot, first average the five-rollout return sequence over held-out tasks within
each seed. Then compute the mean and 95% confidence interval across seeds independently at each
rollout index. Do not treat individual tasks as independent random seeds when computing the shaded
confidence interval.


In [ ]:
def _shared_frame_grid(
    runs: Sequence[Mapping[str, Any]],
    points: int = 300,
) -> np.ndarray:
    starts = [float(np.asarray(run["frames"])[0]) for run in runs]
    ends = [float(np.asarray(run["frames"])[-1]) for run in runs]
    low = max(starts)
    high = min(ends)
    if not high > low:
        raise ValueError("Seed runs do not share an overlapping frame range.")
    return np.linspace(low, high, points)


def _aggregate_curve(
    runs: Sequence[Mapping[str, Any]],
    key: str,
    points: int = 300,
    smoothing_window: int = 1,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    if not runs:
        raise ValueError("No runs were supplied.")

    for run in runs:
        validate_history(run)

    grid = _shared_frame_grid(runs, points=points)
    interpolated = []

    for run in runs:
        frames = np.asarray(run["frames"], dtype=np.float64)
        values = np.asarray(run[key], dtype=np.float64)

        if smoothing_window > 1:
            values = (
                pd.Series(values)
                .rolling(smoothing_window, min_periods=1, center=True)
                .mean()
                .to_numpy()
            )

        interpolated.append(np.interp(grid, frames, values))

    samples = np.stack(interpolated, axis=0)
    mean = samples.mean(axis=0)

    if samples.shape[0] == 1:
        ci95 = np.zeros_like(mean)
    else:
        sem = samples.std(axis=0, ddof=1) / np.sqrt(samples.shape[0])
        ci95 = 1.96 * sem

    return grid, mean, ci95

In [ ]:
def plot_figure6_halfcheetahvel(
    histories: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    smoothing_window: int = 1,
    interpolation_points: int = 300,
    show_individual_seeds: bool = False,
) -> None:
    """
    Plot a Figure 6-style HalfCheetahVel comparison.

    Top: first-rollout return.
    Bottom: adapted/N-th-rollout return.
    """
    nonempty = {name: runs for name, runs in histories.items() if len(runs) > 0}
    if not nonempty:
        raise ValueError("No completed runs were found in histories.")

    fig, axes = plt.subplots(2, 1, figsize=(10, 9), sharex=True)

    panels = [
        ("first_rollout_return", "First rollout"),
        ("adapted_return", "Adapted / method-specific final rollout"),
    ]

    for axis, (key, title) in zip(axes, panels):
        for method, runs in nonempty.items():
            grid, mean, ci95 = _aggregate_curve(
                runs,
                key,
                points=interpolation_points,
                smoothing_window=smoothing_window,
            )

            line = axis.plot(grid, mean, label=method)[0]
            axis.fill_between(grid, mean - ci95, mean + ci95, alpha=0.20)

            if show_individual_seeds:
                for run in runs:
                    axis.plot(
                        np.asarray(run["frames"]),
                        np.asarray(run[key]),
                        alpha=0.20,
                        linewidth=0.8,
                        color=line.get_color(),
                    )

        axis.set_title(title)
        axis.set_ylabel("Average undiscounted return")
        axis.grid(alpha=0.25)
        axis.legend()

    axes[-1].set_xlabel("Meta-training environment frames")
    fig.suptitle("HalfCheetahVel: Figure 6-style meta-RL learning curves")
    fig.tight_layout()
    plt.show()

In [ ]:
def summarize_final_performance(
    histories: Mapping[str, Sequence[Mapping[str, Any]]],
    tail_points: int = 5,
) -> pd.DataFrame:
    """Summarize final performance across seeds without saving files."""
    rows = []

    for method, runs in histories.items():
        if not runs:
            continue

        first_values = []
        adapted_values = []
        final_frames = []

        for run in runs:
            validate_history(run)
            first = np.asarray(run["first_rollout_return"], dtype=np.float64)
            adapted = np.asarray(run["adapted_return"], dtype=np.float64)
            frames = np.asarray(run["frames"], dtype=np.float64)

            k = min(tail_points, len(first))
            first_values.append(float(first[-k:].mean()))
            adapted_values.append(float(adapted[-k:].mean()))
            final_frames.append(float(frames[-1]))

        first_values = np.asarray(first_values)
        adapted_values = np.asarray(adapted_values)

        rows.append(
            {
                "method": method,
                "seeds": len(runs),
                "final_frames_mean": np.mean(final_frames),
                "first_rollout_mean": first_values.mean(),
                "first_rollout_std": first_values.std(ddof=1) if len(first_values) > 1 else 0.0,
                "adapted_mean": adapted_values.mean(),
                "adapted_std": adapted_values.std(ddof=1) if len(adapted_values) > 1 else 0.0,
                "mean_adaptation_gain": (adapted_values - first_values).mean(),
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=[
                "method",
                "seeds",
                "final_frames_mean",
                "first_rollout_mean",
                "first_rollout_std",
                "adapted_mean",
                "adapted_std",
                "mean_adaptation_gain",
            ]
        )

    return pd.DataFrame(rows).sort_values("adapted_mean", ascending=False)

In [ ]:
def plot_rollout_adaptation(
    rollout_results: Mapping[str, np.ndarray],
) -> None:
    """
    Plot return versus within-task rollout.

    Each value must have shape [num_seeds, num_tasks, num_rollouts].
    This diagnostic is additional to the required Figure 6-style plot.
    """
    fig, axis = plt.subplots(figsize=(9, 5))

    for method, values in rollout_results.items():
        array = np.asarray(values, dtype=np.float64)
        if array.ndim != 3:
            raise ValueError(
                f"{method} must have shape [seed, task, rollout], got {array.shape}."
            )

        per_seed = array.mean(axis=1)
        mean = per_seed.mean(axis=0)

        if per_seed.shape[0] == 1:
            ci95 = np.zeros_like(mean)
        else:
            ci95 = 1.96 * per_seed.std(axis=0, ddof=1) / np.sqrt(per_seed.shape[0])

        x = np.arange(1, array.shape[-1] + 1)
        axis.plot(x, mean, marker="o", label=method)
        axis.fill_between(x, mean - ci95, mean + ci95, alpha=0.20)

    axis.set_xlabel("Within-task rollout")
    axis.set_ylabel("Average undiscounted return")
    axis.set_title("Held-out HalfCheetahVel adaptation")
    axis.grid(alpha=0.25)
    axis.legend()
    plt.show()

In [ ]:
def validate_figure4_results(
    rollout_results: Mapping[str, np.ndarray],
    required_rollouts: int = FIGURE4_EVALUATION_ROLLOUTS,
) -> None:
    """Validate arrays used for the Figure 4-style HalfCheetahVel plot."""
    if not rollout_results:
        raise ValueError("No Figure 4 rollout results were provided.")

    for method, values in rollout_results.items():
        array = np.asarray(values, dtype=np.float64)
        if array.ndim != 3:
            raise ValueError(
                f"{method} must have shape [seed, task, rollout], got {array.shape}."
            )
        if array.shape[0] < 1 or array.shape[1] < 1:
            raise ValueError(f"{method} must contain at least one seed and one task.")
        if array.shape[2] < required_rollouts:
            raise ValueError(
                f"{method} has only {array.shape[2]} rollouts; "
                f"Figure 4 requires at least {required_rollouts}."
            )
        if not np.isfinite(array[:, :, :required_rollouts]).all():
            raise ValueError(f"{method} contains non-finite Figure 4 returns.")


def plot_figure4_halfcheetahvel(
    rollout_results: Mapping[str, np.ndarray],
    *,
    required_rollouts: int = FIGURE4_EVALUATION_ROLLOUTS,
    show_individual_seeds: bool = False,
) -> None:
    """
    Plot a Figure 4-style HalfCheetahVel meta-test comparison.

    Each input array has shape [num_seeds, num_test_tasks, num_rollouts].
    The first five rollouts are used by default. Task returns are averaged
    within each seed before the across-seed mean and confidence interval.
    """
    validate_figure4_results(rollout_results, required_rollouts)

    fig, axis = plt.subplots(figsize=(9, 5))
    x = np.arange(1, required_rollouts + 1)

    for method, values in rollout_results.items():
        array = np.asarray(values, dtype=np.float64)[:, :, :required_rollouts]
        per_seed = array.mean(axis=1)
        mean = per_seed.mean(axis=0)

        if per_seed.shape[0] == 1:
            ci95 = np.zeros_like(mean)
        else:
            ci95 = 1.96 * per_seed.std(axis=0, ddof=1) / np.sqrt(per_seed.shape[0])

        line = axis.plot(x, mean, marker="o", label=method)[0]
        axis.fill_between(
            x,
            mean - ci95,
            mean + ci95,
            alpha=0.20,
            color=line.get_color(),
        )

        if show_individual_seeds:
            for seed_curve in per_seed:
                axis.plot(
                    x,
                    seed_curve,
                    alpha=0.20,
                    linewidth=0.8,
                    color=line.get_color(),
                )

    axis.set_xticks(x)
    axis.set_xlabel("Episode / rollout")
    axis.set_ylabel("Average undiscounted return")
    axis.set_title("HalfCheetahVel: Figure 4-style average test performance")
    axis.grid(alpha=0.25)
    axis.legend()
    fig.tight_layout()
    plt.show()


## Optional plotting smoke test

This synthetic example only verifies that your plotting environment works. It is not an expected
result and must not be included as experimental evidence.

In [ ]:
def make_synthetic_plotting_demo() -> Dict[str, List[Dict[str, Any]]]:
    demo: Dict[str, List[Dict[str, Any]]] = {
        "VariBAD": [],
        "RL2": [],
        "MAML": [],
    }
    frames = np.linspace(1_000, 1_000_000, 40)

    for method_index, method in enumerate(demo):
        for seed in range(3):
            rng = np.random.default_rng(100 * method_index + seed)
            base = -280 + 210 * (1 - np.exp(-frames / (250_000 + 50_000 * method_index)))
            first = base + rng.normal(0, 7, size=len(frames))
            adapted = base + 30 - 8 * method_index + rng.normal(0, 7, size=len(frames))

            demo[method].append(
                {
                    "method": method,
                    "seed": seed,
                    "frames": frames.copy(),
                    "first_rollout_return": first,
                    "adapted_return": adapted,
                    "metadata": {"synthetic_demo": True},
                }
            )

    return demo


# Uncomment to test the plotting utilities.
# synthetic_histories = make_synthetic_plotting_demo()
# plot_figure6_halfcheetahvel(
#     synthetic_histories,
#     smoothing_window=3,
#     show_individual_seeds=True,
# )
# display(summarize_final_performance(synthetic_histories))

In [ ]:
def make_synthetic_figure4_demo() -> Dict[str, np.ndarray]:
    """Create synthetic arrays only for testing the Figure 4 plotting utility."""
    demo: Dict[str, np.ndarray] = {}
    num_seeds = 3
    num_tasks = 8
    num_rollouts = FIGURE4_EVALUATION_ROLLOUTS

    for method_index, method in enumerate(("VariBAD", "RL2", "MAML")):
        values = np.empty((num_seeds, num_tasks, num_rollouts), dtype=np.float64)
        for seed in range(num_seeds):
            rng = np.random.default_rng(1_000 * method_index + seed)
            adaptation = 28.0 - 5.0 * method_index
            for task in range(num_tasks):
                start = -190.0 + rng.normal(0.0, 8.0)
                trend = adaptation * np.arange(num_rollouts)
                values[seed, task] = start + trend + rng.normal(0.0, 5.0, num_rollouts)
        demo[method] = values

    return demo


# Uncomment to test the Figure 4 plotting utility.
# synthetic_figure4_results = make_synthetic_figure4_demo()
# plot_figure4_halfcheetahvel(
#     synthetic_figure4_results,
#     show_individual_seeds=True,
# )


# Part VII - Final evaluation and figures


### Figure 4 final-checkpoint evaluation

After selecting checkpoints without using the test tasks, run a separate five-rollout evaluation for
Figure 4. The rollout sequence must be continuous with respect to method-specific adaptation state,
even though the physical environment resets at each episode boundary. Use the exact same selected
checkpoint for every rollout of a task.


In [ ]:
# Figure 6-style comparison from the actual training histories.

plot_figure6_halfcheetahvel(
    all_histories,
    smoothing_window=3,
    show_individual_seeds=True,
)

final_summary = summarize_final_performance(
    all_histories, tail_points=5
)
display(final_summary)


In [ ]:
# Collect five continuous held-out meta-test rollouts from each final checkpoint.
#
# Shapes are [num_seeds, num_test_tasks, 5].
#
# MAML mapping: rollout 1 uses the learned initialization.  After each rollout,
# that rollout becomes the support data for the next inner update, and the
# resulting fast parameters are retained until the task changes.

rollout_results: Dict[str, np.ndarray] = {}

if all_histories["VariBAD"]:
    rollout_results["VariBAD"] = np.stack(
        [
            evaluate_varibad_rollouts_from_history(
                run,
                CONFIG,
                TASKS.test,
                num_rollouts=FIGURE4_EVALUATION_ROLLOUTS,
            )
            for run in all_histories["VariBAD"]
        ],
        axis=0,
    )

if all_histories["RL2"]:
    rollout_results["RL2"] = np.stack(
        [
            evaluate_rl2_rollouts_from_history(
                run,
                CONFIG,
                TASKS.test,
                num_rollouts=FIGURE4_EVALUATION_ROLLOUTS,
            )
            for run in all_histories["RL2"]
        ],
        axis=0,
    )

if all_histories["MAML"]:
    rollout_results["MAML"] = np.stack(
        [
            evaluate_maml_rollouts_from_history(
                run,
                CONFIG,
                TASKS.test,
                num_rollouts=FIGURE4_EVALUATION_ROLLOUTS,
            )
            for run in all_histories["MAML"]
        ],
        axis=0,
    )

for method, array in rollout_results.items():
    print(method, array.shape)

plot_rollout_adaptation(rollout_results)


In [ ]:
# Required Figure 4-style HalfCheetahVel panel.

validate_figure4_results(rollout_results)
plot_figure4_halfcheetahvel(
    rollout_results,
    show_individual_seeds=True,
)


# Question 1: Three Mechanisms for Fast Adaptation

The implementation and figures above provide the trained-model evidence used in this discussion.
The next cell summarizes the **selected final validation checkpoint** and the existing five-rollout
held-out test evaluation. It does not perform additional training or manually enter results.

In [ ]:
# Evidence for Question 1 from the trained checkpoints already produced above.
# This uses the last validation checkpoint in each history (not a manually
# entered number) plus the existing five-rollout test arrays.

question1_validation_rows = []
for method, runs in all_histories.items():
    if not runs:
        continue
    first = np.asarray(
        [float(run["first_rollout_return"][-1]) for run in runs],
        dtype=np.float64,
    )
    adapted = np.asarray(
        [float(run["adapted_return"][-1]) for run in runs],
        dtype=np.float64,
    )
    question1_validation_rows.append(
        {
            "method": method,
            "seeds": len(runs),
            "first_rollout_return": float(first.mean()),
            "adapted_return": float(adapted.mean()),
            "adaptation_gain": float((adapted - first).mean()),
        }
    )

question1_validation_evidence = pd.DataFrame(question1_validation_rows)
if not question1_validation_evidence.empty:
    question1_validation_evidence = question1_validation_evidence.sort_values(
        "adapted_return", ascending=False
    ).reset_index(drop=True)

question1_rollout_rows = []
for method, values in rollout_results.items():
    array = np.asarray(values, dtype=np.float64)
    per_rollout = array.mean(axis=(0, 1))
    row = {"method": method}
    for rollout_index, value in enumerate(per_rollout, start=1):
        row[f"rollout_{rollout_index}"] = float(value)
    row["rollout_2_minus_1"] = float(per_rollout[1] - per_rollout[0])
    row["rollout_5_minus_1"] = float(per_rollout[4] - per_rollout[0])
    question1_rollout_rows.append(row)

question1_rollout_evidence = pd.DataFrame(question1_rollout_rows)

print("Final validation checkpoint evidence")
display(question1_validation_evidence)
print("Five-rollout held-out test evidence")
display(question1_rollout_evidence)

if CONFIG is DEBUG_CONFIG or CONFIG.total_frames == DEBUG_CONFIG.total_frames:
    print(
        "Note: CONFIG is the short DEBUG_CONFIG. These values are valid evidence "
        "from the trained checkpoints, but they should not be interpreted as "
        "converged full-scale meta-RL performance."
    )

## Question 1: Three Mechanisms for Fast Adaptation

### 1.

- **MAML:** Task-specific information is stored in the adapted fast parameters \(\theta'\). It is updated by policy-gradient inner-loop steps using support trajectories. When moving to a new task, the fast parameters are discarded and reset to the learned meta-initialization \(\theta\).
- **RL2:** Task-specific information is stored in the recurrent hidden state. It is updated at every timestep from the observation, previous action, reward, and done/reset flag. The hidden state is preserved across episodes of the same task and reset only when the task changes.
- **VariBAD:** Task-specific information is stored in the inferred posterior \(q_\phi(m\mid\tau_{:t})\) and recurrent encoder state. The belief is updated after each transition. At a new task, the encoder state is reset and the posterior returns to its prior; it is preserved across episodes of the same task.

### 2.

All three methods can use early experience to improve later behavior. MAML uses the first rollout to obtain gradients for fast adaptation; RL2 can learn to gather useful task information through its recurrent memory; VariBAD explicitly updates a probabilistic belief about the hidden task.

VariBAD represents uncertainty most explicitly because it maintains a posterior distribution over the latent task. This does not guarantee the best return because good inference alone is insufficient: the latent representation, reward model, and policy must also be learned accurately and the policy must use the inferred belief effectively.

In this debug run, the final validation adaptation gains were approximately:

- RL2: \(-29.19\)
- MAML: \(-19.77\)
- VariBAD: \(+4.33\)

Thus, only VariBAD showed a positive second-rollout gain at the final validation checkpoint. However, the five-rollout held-out test showed broader within-task improvement for RL2 and MAML: rollout 5 minus rollout 1 was about \(+8.13\) for RL2 and \(+6.24\) for MAML, while VariBAD was about \(-4.15\). These results are from the short debug configuration, so they should be treated as evidence of the implemented adaptation behavior rather than converged performance.

### 3.

If the adaptation state were reset between two episodes of the same task:

- **MAML:** the adapted parameters would be lost, so the next rollout would behave more like an unadapted first rollout.
- **RL2:** resetting the recurrent hidden state would erase information learned from the first episode, so the second rollout would again behave like an initial rollout.
- **VariBAD:** resetting the encoder/posterior would return the belief to its prior, forcing the agent to infer the hidden task again from scratch.

Therefore, an incorrect reset would mainly reduce the adapted-return advantage and make later-rollout performance move closer to first-rollout performance. The first rollout itself would be largely unaffected because the information loss occurs only after it.